# `table_dataset_extraction_and_pwc_benchmark_gliner.ipynb`

PwC benchmark: **Datasets** in `data/pwc_final.json`, corpus `data/pdf_files_3`.

Related notebook: `table_dataset_extraction_and_pwc_benchmark_ollama.ipynb` (same three modes, Ollama models).

Replication steps are in `table_extraction/README_pwc_benchmarks.md`.

Prediction setups (summary sheet `dataset_gt_mode_summary` when enabled):

`tables_only` uses LightOnOCR table JSON plus GLiNER (sheet Tables With Datasets).

`text_only` uses GROBID TEI under `data/xml_files_3` plus GLiNER on narrative text; predictions are filtered to the per-paper PwC dataset vocabulary.

`tables_plus_text` is the union of the two modes for each corpus paper.


In [ ]:
# Optional: bump HF stack in THIS kernel for Hub dataset loads
import subprocess
import sys

UPGRADE_HF_LIBS_IN_THIS_KERNEL = True  # Set False once stable to skip reinstalling each run

if UPGRADE_HF_LIBS_IN_THIS_KERNEL:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-U",
            "datasets>=3.0.0",
            "huggingface_hub>=0.26.0",
            "fsspec>=2024.10.0",
        ]
    )
    print("Upgraded with:", sys.executable)
    print("Restart the kernel and re-run from the first imports cell.")
    print("If the error persists after restart, in the same env: conda install -c conda-forge 'fsspec>=2024.10.0'")
else:
    print("Upgrade skipped.")

In [ ]:
from __future__ import annotations

import json
import os
import difflib
import re
import unicodedata
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import pandas as pd

## Optional Hugging Face login

Improves Hub rate limits. In the next code cell set `RUN_HF_LOGIN=True` and paste your [token](https://huggingface.co/settings/tokens) in the **widget** (do not commit tokens in notebook source).

Alternatively: `huggingface-cli login` or `export HF_TOKEN=...` before Jupyter.


In [ ]:
# Optional: Hugging Face login (better Hub rate limits)
RUN_HF_LOGIN = False  # Set True to authenticate on this machine

if RUN_HF_LOGIN:
    from huggingface_hub import login

    try:
        from huggingface_hub import notebook_login

        notebook_login()
    except Exception as exc:
        print("notebook_login unavailable in this frontend; falling back to login():", exc)
        login()
else:
    print(
        "HF login skipped (RUN_HF_LOGIN=False).\n"
        "HF_TOKEN or a prior CLI login is enough.\n"
        "To use the widget: set RUN_HF_LOGIN=True and re-run this cell."
    )

## Pipeline (extraction and evaluation)

Step 1: Configuration. Step 2: Corpus (`CORPUS_PDF_PATHS`). Step 3: Extraction (single code cell). Step 4: Evaluation against `pwc_final.json` (`pdf_files_3`).


In [ ]:
print("[pipeline] pwc_final → corpus → extraction (one cell) → eval")


In [ ]:
# Configuration
import os
import sys
import time

EVAL_OUTPUT_SUFFIX = "_minimal"
BERTSCORE_LANG = "en"
# Minimal pipeline — no GT-assisted extraction aids.
MODE = "raw"
USE_BLACKLIST = False
REQUIRE_PDF_FILES_CORPUS = True
AUTO_RUN_EXTRACTION_IF_MISSING = False  # use inline pipeline in corpus cell
RUN_EXTRACTION_FIRST = False  # legacy; use RUN_INLINE_EXTRACTION
# Corpus pequeño: PDFs + LightOCR caches in data/pdf_files/
# Corpus reducido: PDFs + XML alineados en data/pdf_files_3/ y data/xml_files_3/
PDF_FILES_DIR = Path("data/pdf_files_3")
XML_FILES_DIR = Path("data/xml_files_3")  # GROBID TEI; run build_xml_files_3_corpus.py
GT_XML_SECTION_KEYWORDS = []  # empty → full TEI body (tables stripped)
GT_SCAN_FULL_BODY_IF_EMPTY = True
CORPUS_TAG = "pdf_files"  # suffix for extraction/eval Excel
TABLE_EXTRACTION_DIR = Path("table_extraction")
# Optional Combinations vs Papers with Code (HF dataset); off = skip load/eval, no global_scores sheet
EVALUATE_COMBINATIONS_VS_PWC = False
EVAL_COMBINATIONS_USE_JSON_IDS = True
# PDFs in pdf_files/ with zero rows in Combinations → console only unless True
EXPORT_COVERAGE_MISSING_TO_EXCEL = False

# Three comparable prediction setups (dataset GT eval):
EVAL_SETUP_TABLES_ONLY = True   # LightOnOCR + GLiNER: Tables With Datasets sheet only
EVAL_TEXT_ONLY_USE_GLINER = True  # GLiNER on TEI narrative (not regex-only)
EVAL_SETUP_TEXT_ONLY = True     # GROBID narrative text from data/xml_files_3 only
EVAL_SETUP_TABLES_PLUS_TEXT = True  # union(tables_only, text_only) for all corpus papers
FILTER_PREDICTIONS_TO_PWC_GT_DATASETS = False
EVAL_ONLY_PWC_MATCHED_PAPERS = True  # eval/audit: omit pipeline keys with no GT match

# Ground truth: Datasets in data/pwc_final.json, restricted to selected PDFs in PDF_FILES_DIR
EVALUATE_AGAINST_DATASET_METRICS_GT = True
DATASET_METRICS_GT_JSON_BASENAME = "pwc_final.json"
DATASET_JSON_FOR_IDS = DATASET_METRICS_GT_JSON_BASENAME
GT_FILTER_TO_PDF_CORPUS = True  # only PDFs in folder matched to pwc_final (skip missing)
GT_EVAL_SKIP_EMPTY_DATASETS = True  # exclude empty Datasets from GT
PWC_ORIGIN_JSON_BASENAME = "pwc_final.json"
PWC_SAMPLE_SIZE = None  # e.g. 50 for quick test
PWC_SAMPLE_SEED = 42
PIPELINE_TIMING_SHEET = "pipeline_timing"
SHOW_VERBOSE = False  # OCR progress per PDF
EVAL_DATASET_GT_USE_JSON_IDS = True
DATASET_GT_SHEET_IN_PWC_EXCEL = "dataset_gt_mode_summary"
EXPORT_CORPUS_PWC_MATCH_AUDIT = True


def _combinations_excel_name(mode: str) -> str:
    tag = (CORPUS_TAG or "").strip().replace("/", "_")
    if tag:
        return f"gliner2_lightonocr_dataset_combinations_{mode}_{tag}.xlsx"
    return f"gliner2_lightonocr_dataset_combinations_{mode}.xlsx"


def _evaluation_excel_name() -> str:
    tag = (CORPUS_TAG or "").strip().replace("/", "_")
    return f"evaluation_{tag}.xlsx" if tag else "evaluation_against_pwc.xlsx"


def _resolve_output_excel(repo_root: Path | None = None) -> Path:
    """Evaluation report next to these notebooks (table_extraction/)."""
    if repo_root is None:
        for c in (Path.cwd(), Path.cwd().parent):
            if (c / "table_extraction/evaluation_table_extraction_gliner2_lightonocr.ipynb").exists():
                repo_root = c.resolve()
                break
    if repo_root is not None:
        out = (repo_root / TABLE_EXTRACTION_DIR / _evaluation_excel_name()).resolve()
        out.parent.mkdir(parents=True, exist_ok=True)
        return out
    return Path("table_extraction") / _evaluation_excel_name()



def _resolve_dataset_metrics_gt_output_excel() -> Path:
    for p in (
        Path("evaluation_against_dataset_metrics_gt.xlsx"),
        Path("table_extraction/evaluation_against_dataset_metrics_gt.xlsx"),
    ):
        if p.parent == Path(".") or p.parent.is_dir():
            return p
    return Path("table_extraction/evaluation_against_dataset_metrics_gt.xlsx")


def _find_repo_root_for_extraction() -> Path:
    for c in (Path.cwd(), Path.cwd().parent):
        if (c / "data" / PWC_ORIGIN_JSON_BASENAME).is_file():
            return c.resolve()
        if (c / "data" / PWC_ORIGIN_JSON_BASENAME).is_file():
            return c.resolve()
    raise FileNotFoundError(
        "Repository root not found (missing data/pwc_final.json)."
    )


def _resolve_pdf_files_dir(repo_root: Path | None = None) -> Path:
    """PDF corpus + LightOCR JSON caches (this notebook: data/pdf_files/)."""
    root = (repo_root or _find_repo_root_for_extraction()).resolve()
    pdf_dir = (root / PDF_FILES_DIR).resolve()
    if not pdf_dir.is_dir():
        raise FileNotFoundError(
            f"PDF corpus not found: {pdf_dir}. Put *.pdf files there."
        )
    return pdf_dir


def _resolve_table_extraction_dir(repo_root: Path | None = None) -> Path:
    """Notebook folder: extraction + evaluation Excel files."""
    root = (repo_root or _find_repo_root_for_extraction()).resolve()
    out_dir = (root / TABLE_EXTRACTION_DIR).resolve()
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir


_PIPELINE_TIMING: dict[str, float] = {}


def _resolve_input_excel(mode: str, require_pdf_files: bool = True) -> Path:
    fname = _combinations_excel_name(mode)
    repo_root = _find_repo_root_for_extraction()
    excel_dir = _resolve_table_extraction_dir(repo_root)

    p = excel_dir / fname
    if p.is_file():
        return p.resolve()

    tag = (CORPUS_TAG or "").strip()
    found = sorted(
        (
            x
            for x in excel_dir.glob("gliner2_lightonocr_dataset_combinations_*.xlsx")
            if not tag or tag.replace("/", "_") in x.name
        ),
        key=lambda x: x.stat().st_mtime,
        reverse=True,
    )
    if found:
        return found[0].resolve()

    if require_pdf_files and AUTO_RUN_EXTRACTION_IF_MISSING:
        _run_extraction_over_pdf_files()
        p = excel_dir / fname
        if p.is_file():
            return p.resolve()
        found = sorted(
            excel_dir.glob("gliner2_lightonocr_dataset_combinations_*.xlsx"),
            key=lambda x: x.stat().st_mtime,
            reverse=True,
        )
        if found:
            return found[0].resolve()

    if require_pdf_files:
        raise FileNotFoundError(
            f"Could not find {fname} in {excel_dir} after auto-extraction."
        )

    raise FileNotFoundError(f"Could not find {fname} in {excel_dir}.")


INPUT_EXCEL = None  # set in corpus + extraction cell
INPUT_SHEET = "Combinations"
OUTPUT_EXCEL = _resolve_output_excel(_find_repo_root_for_extraction())
TASK_FILTER = "link prediction"  # "" = all tasks in the dataset
MATCH_THRESHOLD = 0.50

print(f"PDF_FILES_DIR = {_resolve_pdf_files_dir(_find_repo_root_for_extraction())}")
print(f"TABLE_EXTRACTION_DIR = {_resolve_table_extraction_dir(_find_repo_root_for_extraction())}")
print(f"OUTPUT_EXCEL = {OUTPUT_EXCEL}")
print(
    f"Minimal pipeline: raw GLiNER, BERTScore (lang={BERTSCORE_LANG}), outputs *{EVAL_OUTPUT_SUFFIX}.xlsx"
)
print(f"GT: data/{PWC_ORIGIN_JSON_BASENAME} Metrics≠[] → {PDF_FILES_DIR}")



In [ ]:
# No METRIC_ALIASES — eval uses normalize_dataset() / normalize_metric() only.

PAPER_COL_CANDIDATES = ["paper_title", "paper", "title", "paper_name", "paper_url", "url"]
DATASET_COL_CANDIDATES = ["dataset", "dataset_name", "eval_dataset", "benchmark"]
METRIC_COL_CANDIDATES = ["metric", "metric_name", "evaluation_metric", "metrics"]
TASK_COL_CANDIDATES = ["task", "task_name", "subtask"]



In [ ]:
def _strip_accents(text: str) -> str:
    return "".join(ch for ch in unicodedata.normalize("NFKD", text) if not unicodedata.combining(ch))


def normalize_text(text: object) -> str:
    if text is None:
        return ""
    s = str(text).strip().lower()
    s = _strip_accents(s)
    s = re.sub(r"\s+", " ", s)
    return s


def normalize_paper(text: object) -> str:
    s = normalize_text(text)
    if s.startswith("http"):
        s = s.rstrip("/")
        s = s.split("/")[-1]
    s = re.sub(r"[^a-z0-9]+", " ", s).strip()
    return s


def normalize_dataset(text: object) -> str:
    s = normalize_text(text)
    s = re.sub(r"\([^)]*\)", "", s).strip()
    s = s.replace(" ", "").replace("_", "").replace("-", "")
    return s


def _strip_trailing_plural(s: str) -> str:
    """Drop one trailing 's' on purely alphabetic tokens (hits@10 unchanged)."""
    if not s or "@" in s or not s.isalpha():
        return s
    if len(s) > 3 and s.endswith("s") and not s.endswith("ss"):
        return s[:-1]
    return s


def normalize_metric(text: object) -> str:
    """Eval normalization only — no aliases, no regex canon."""
    s = normalize_text(text)
    if not s:
        return ""
    s = re.sub(r"[^a-z0-9@]+", "", s)
    return _strip_trailing_plural(s)




def safe_f1(p: float, r: float) -> float:
    return 0.0 if (p + r) == 0 else (2 * p * r) / (p + r)


def first_present_column(df: pd.DataFrame, candidates: Iterable[str]) -> str | None:
    cols = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in cols:
            return cols[c.lower()]
    return None


In [ ]:
# Ground truth: data/pwc_final.json (read-only; no JSON enrichment).
print("GT: data/pwc_final.json (no enrichment)")


In [ ]:

def _pwc_datasets_list(obj: dict) -> list:
    """Dataset names from pwc_final (`Datasets` or legacy `datasets`)."""
    raw = obj.get("datasets")
    if raw is None or (isinstance(raw, list) and len(raw) == 0):
        raw = obj.get("Datasets")
    if not isinstance(raw, list):
        return []
    return [str(m).strip() for m in raw if str(m).strip()]


def load_ours_combinations(path: Path, sheet_name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Input Excel not found: {path}")
    df = pd.read_excel(path, sheet_name=sheet_name)
    colmap = {c.lower(): c for c in df.columns}
    missing = {"paper", "dataset", "metric"} - set(colmap.keys())
    if missing:
        raise ValueError(f"Sheet '{sheet_name}' missing columns: {missing}")
    out = pd.DataFrame({
        "paper_raw": df[colmap["paper"]],
        "dataset_raw": df[colmap["dataset"]],
        "metric_raw": df[colmap["metric"]],
    })
    out["paper_norm"] = out["paper_raw"].map(normalize_paper)
    out["dataset_norm"] = out["dataset_raw"].map(normalize_dataset)
    out["metric_norm"] = out["metric_raw"].map(normalize_metric)
    out = out[(out["paper_norm"] != "") & (out["dataset_norm"] != "")]
    return out.drop_duplicates()


def _hf_dataset_token() -> str | None:
    """HF token from the environment if set.
    If None, `load_dataset` uses the Hub default (cached CLI / notebook login when present)."""
    tok = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
    return tok.strip() if tok and str(tok).strip() else None

def _pwc_as_sequence(x) -> list:
    if x is None:
        return []
    if hasattr(x, "tolist"):
        try:
            x = x.tolist()
        except Exception:
            return []
    if isinstance(x, (list, tuple)):
        return list(x)
    return [x]


def _pwc_walk_dataset_nodes(d: dict):
    yield d
    for sd in _pwc_as_sequence(d.get("subdatasets")):
        if isinstance(sd, dict):
            yield from _pwc_walk_dataset_nodes(sd)


def _arxiv_norm_from_text(text: object) -> str:
    if text is None:
        return ""
    s = str(text).strip().lower()
    m = re.search(r"(\d{4}\.\d{4,5})(?:v\d+)?", s)
    return m.group(1) if m else ""


def _paperswithcode_slug_from_url(url: object) -> str:
    if url is None:
        return ""
    s = str(url).strip().rstrip("/")
    if "paperswithcode.com/paper/" in s:
        return s.split("paperswithcode.com/paper/", 1)[-1].split("/")[0].strip().lower()
    return ""


def _pdf_stem_slug_from_title(title: object, max_len: int = 50) -> str:
    """Slug like pdf_files_* stems: normalized title → words joined with '_' (truncated)."""
    pn = normalize_paper(title)
    if not pn:
        return ""
    return re.sub(r"\s+", "_", pn)[:max_len]


def _stem_matches_title_slug(stem: str, title_slug: str) -> bool:
    if not stem or not title_slug:
        return False
    a, b = stem.lower(), title_slug.lower()
    return a == b or b.startswith(a) or a.startswith(b)


def _pwc_identity_from_row(mrow: dict | None, paper_raw: str) -> tuple[str, str]:
    """Stable paper key aligned with dataset JSON: arxiv > PapersWithCode slug > normalized title."""
    mr = mrow if isinstance(mrow, dict) else {}
    for k in ("arxiv_id", "arxiv", "paper_arxiv_id"):
        a = _arxiv_norm_from_text(mr.get(k))
        if a:
            return f"arxiv:{a}", "arxiv"
    for ukey in ("paper_url", "url"):
        sl = _paperswithcode_slug_from_url(mr.get(ukey))
        if sl:
            return f"slug:{sl}", "slug"
    blob = " ".join(str(mr.get(x, "") or "") for x in ("paper_title", "paper", "paper_url"))
    mb = re.search(r"\b(\d{4}\.\d{5})\b", blob) or re.search(r"\b(\d{4}\.\d{4})\b", blob)
    if mb:
        return f"arxiv:{mb.group(1)}", "arxiv"
    a2 = _arxiv_norm_from_text(paper_raw)
    if a2:
        return f"arxiv:{a2}", "arxiv"
    sl2 = _paperswithcode_slug_from_url(paper_raw)
    if sl2:
        return f"slug:{sl2}", "slug"
    return f"title:{normalize_paper(paper_raw)}", "title"


def _dataset_json_identity(o: dict) -> tuple[str, str]:
    for k in ("arxiv_id", "url_abs", "url_pdf", "paper_url"):
        a = _arxiv_norm_from_text(o.get(k))
        if a:
            return f"arxiv:{a}", "arxiv"
    sl = _paperswithcode_slug_from_url(o.get("paper_url"))
    if sl:
        return f"slug:{sl}", "slug"
    return f"title:{normalize_paper(o.get('title'))}", "title"


def _pwc_final_lookup_maps(papers: list) -> tuple[dict, dict, dict, dict, dict]:
    by_arxiv: dict[str, dict] = {}
    by_slug: dict[str, dict] = {}
    by_title: dict[str, dict] = {}
    by_pdf_stem: dict[str, dict] = {}
    by_title_stem: dict[str, dict] = {}
    for o in papers:
        if not isinstance(o, dict):
            continue
        for k in ("arxiv_id", "url_abs", "url_pdf"):
            a = _arxiv_norm_from_text(o.get(k))
            if a:
                by_arxiv.setdefault(a, o)
        sl = _paperswithcode_slug_from_url(o.get("paper_url"))
        if sl:
            by_slug.setdefault(sl, o)
        pn = normalize_paper(o.get("title"))
        if pn:
            by_title.setdefault(pn, o)
        ts = _pdf_stem_slug_from_title(o.get("title"))
        if ts:
            by_title_stem.setdefault(ts, o)
        lp = str(o.get("local_pdf_path") or "").replace("\\", "/")
        if lp:
            by_pdf_stem.setdefault(Path(lp).stem, o)
    return by_arxiv, by_slug, by_title, by_pdf_stem, by_title_stem


def _match_pwc_entry_for_pdf_stem(
    stem: str,
    by_arxiv: dict,
    by_title: dict,
    by_pdf_stem: dict,
    threshold: float,
    *,
    by_title_stem: dict | None = None,
) -> tuple[dict | None, str]:
    if stem in by_pdf_stem:
        return by_pdf_stem[stem], "local_pdf_path"
    a = _arxiv_norm_from_text(stem)
    if a and a in by_arxiv:
        return by_arxiv[a], "arxiv_id"
    by_title_stem = by_title_stem or {}
    stem_l = stem.lower()
    stem_slug = _pdf_stem_slug_from_title(stem.replace("_", " "))
    if stem in by_title_stem:
        return by_title_stem[stem], "title_stem_exact"
    for slug, obj in by_title_stem.items():
        if slug.lower() == stem_l or (stem_slug and slug.lower() == stem_slug.lower()):
            return obj, "title_stem_exact"
    prefix_hits: list[tuple[int, dict, str]] = []
    for slug, obj in by_title_stem.items():
        if _stem_matches_title_slug(stem_slug or stem, slug):
            prefix_hits.append((len(slug), obj, slug))
    if prefix_hits:
        prefix_hits.sort(key=lambda x: x[0], reverse=True)
        best_len, best_o, best_slug = prefix_hits[0]
        ref = (stem_slug or stem).lower()
        if best_slug.lower().startswith(ref) or len(prefix_hits) == 1:
            return best_o, "title_stem_prefix"
    pn = normalize_paper(stem.replace("_", " "))
    if pn in by_title:
        return by_title[pn], "title_exact"
    best_o, best_sc = None, 0.0
    for t, o in by_title.items():
        sc = difflib.SequenceMatcher(None, pn, t).ratio()
        if sc > best_sc:
            best_sc, best_o = sc, o
    if best_o and best_sc >= threshold:
        return best_o, "title_fuzzy"
    return None, "unmatched"


def build_ground_truth_for_pdf_corpus(
    repo_root: Path,
    json_basename: str,
    pdf_rel_dir: Path,
    *,
    match_threshold: float,
    skip_empty_datasets: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """GT dataset names from JSON `Datasets`, one row per PDF in the corpus folder."""
    jp = (repo_root / "data" / json_basename).resolve()
    if not jp.is_file():
        raise FileNotFoundError(jp)
    with open(jp, encoding="utf-8") as f:
        papers = json.load(f)
    if not isinstance(papers, list):
        raise ValueError(f"Expected a JSON list in {jp}")

    pdf_dir = (repo_root / pdf_rel_dir).resolve()
    if not pdf_dir.is_dir():
        raise FileNotFoundError(pdf_dir)

    by_arxiv, _by_slug, by_title, by_pdf_stem, by_title_stem = _pwc_final_lookup_maps(papers)
    dataset_rows: list[dict] = []
    audit_rows: list[dict] = []
    corpus_rows: list[dict] = []

    for pdf_path in sorted(pdf_dir.glob("*.pdf")):
        stem = pdf_path.stem
        obj, how = _match_pwc_entry_for_pdf_stem(
            stem, by_arxiv, by_title, by_pdf_stem, match_threshold,
            by_title_stem=by_title_stem,
        )
        if obj is None:
            corpus_rows.append(
                {
                    "pdf_stem": stem,
                    "pwc_match": "unmatched",
                    "match_method": "",
                    "title": "",
                    "arxiv_id": "",
                    "n_gt_datasets": 0,
                }
            )
            continue
        title = str(obj.get("title") or "").strip()
        pn = normalize_paper(title)
        dsets: set[str] = set()
        for m in _pwc_datasets_list(obj):
            nm = normalize_dataset(str(m))
            if nm:
                dsets.add(nm)
        corpus_rows.append(
            {
                "pdf_stem": stem,
                "pwc_match": "matched",
                "match_method": how,
                "title": title,
                "arxiv_id": str(obj.get("arxiv_id") or ""),
                "n_gt_datasets": len(dsets),
            }
        )
        audit_rows.append(
            {
                "title": title,
                "paper_norm": pn,
                "arxiv_id": str(obj.get("arxiv_id") or ""),
                "pdf_stem": stem,
                "n_gt_datasets": len(dsets),
                "gt_datasets_flat": ", ".join(sorted(dsets)),
            }
        )
        if skip_empty_datasets and not dsets:
            continue
        if not pn:
            continue
        mk, mk_kind = _dataset_json_identity(obj)
        for d in sorted(dsets):
            dataset_rows.append(
                {
                    "paper_raw": title,
                    "dataset_raw": d,
                    "metric_raw": "",
                    "paper_norm": pn,
                    "dataset_norm": d,
                    "paper_match_key": mk,
                    "paper_match_kind": mk_kind,
                    "pdf_stem": stem,
                }
            )

    cols = [
        "paper_raw",
        "dataset_raw",
        "metric_raw",
        "paper_norm",
        "dataset_norm",
        "paper_match_key",
        "paper_match_kind",
        "pdf_stem",
    ]
    gt_df = pd.DataFrame(dataset_rows).drop_duplicates() if dataset_rows else pd.DataFrame(columns=cols)
    return gt_df, pd.DataFrame(audit_rows), pd.DataFrame(corpus_rows)


def _pwc_flatten_nested_eval_tables(df: pd.DataFrame, task_filter: str = "") -> pd.DataFrame:
    """Flatten pwc-archive/evaluation-tables nested parquet (task -> datasets[] -> sota.rows[])."""
    rows_out: list[dict[str, object]] = []
    tf_full = normalize_text(task_filter) if task_filter else ""

    for _, r in df.iterrows():
        task_name = r.get("task", "")
        if tf_full:
            if tf_full not in normalize_text(str(task_name)):
                continue

        for d in _pwc_as_sequence(r.get("datasets")):
            if not isinstance(d, dict):
                continue
            for node in _pwc_walk_dataset_nodes(d):
                ds_name = str(node.get("dataset") or "").strip()
                sota = node.get("sota") if isinstance(node.get("sota"), dict) else None
                if not sota:
                    continue
                for mrow in _pwc_as_sequence(sota.get("rows")):
                    if not isinstance(mrow, dict):
                        continue
                    paper_title = (
                        mrow.get("paper_title") or mrow.get("paper") or mrow.get("paper_url") or ""
                    )
                    paper_title = str(paper_title).strip()
                    mk, mk_kind = _pwc_identity_from_row(mrow, paper_title)
                    met = mrow.get("metrics")
                    if not isinstance(met, dict):
                        continue
                    for mn, mv in met.items():
                        if mv is None or str(mn).strip() == "":
                            continue
                        rows_out.append({
                            "paper_raw": paper_title,
                            "dataset_raw": ds_name,
                            "metric_raw": mn,
                            "task_raw": task_name,
                            "paper_match_key": mk,
                            "paper_match_kind": mk_kind,
                        })

    if not rows_out:
        raise RuntimeError(
            "No rows left after flattening pwc-archive/evaluation-tables "
            "(check TASK_FILTER or dataset revision)."
        )
    flat = pd.DataFrame(rows_out)
    flat["paper_norm"] = flat["paper_raw"].map(normalize_paper)
    flat["dataset_norm"] = flat["dataset_raw"].map(normalize_dataset)
    flat["metric_norm"] = flat["metric_raw"].map(normalize_metric)
    flat = flat[(flat["paper_norm"] != "") & (flat["dataset_norm"] != "")]
    return flat.drop_duplicates()


def load_pwc_dataset(task_filter: str = "") -> pd.DataFrame:
    from datasets import load_dataset

    try:
        ds = load_dataset(
            "pwc-archive/evaluation-tables",
            split="train",
            token=_hf_dataset_token(),
        )
    except KeyError as e:
        if "maxdepth" in str(e).lower() or e.args == ("maxdepth",):
            raise RuntimeError(
                "Failed to load dataset from Hub (KeyError 'maxdepth'). "
                "Upgrade: pip install -U \"datasets>=3.0.0\" \"huggingface_hub>=0.26.0\" \"fsspec>=2024.10.0\" "
                "then restart the kernel."
            ) from e
        raise
    df = ds.to_pandas()
    if df.empty:
        raise RuntimeError("PwC dataset loaded but appears empty")

    # Nested schema (HF snapshot): task, datasets[], sota.rows[].metrics{}
    if "datasets" in df.columns and "task" in df.columns:
        return _pwc_flatten_nested_eval_tables(df, task_filter=task_filter)

    # Flat tabular schema (legacy snapshot)
    paper_col = first_present_column(df, PAPER_COL_CANDIDATES)
    metric_col = first_present_column(df, METRIC_COL_CANDIDATES)
    dataset_col = first_present_column(df, DATASET_COL_CANDIDATES)
    task_col = first_present_column(df, TASK_COL_CANDIDATES)
    if paper_col is None or metric_col is None:
        raise RuntimeError(
            "Unrecognized PwC schema. Columns: "
            f"{list(df.columns)}"
        )
    if task_filter and task_col is not None:
        tf = normalize_text(task_filter)
        keep = df[task_col].astype(str).map(normalize_text).str.contains(tf, na=False)
        df = df[keep].copy()
    if dataset_col is None:
        df["__dataset_fallback__"] = ""
        dataset_col = "__dataset_fallback__"
    out = pd.DataFrame({
        "paper_raw": df[paper_col],
        "dataset_raw": df[dataset_col],
        "metric_raw": df[metric_col],
    })
    out["paper_norm"] = out["paper_raw"].map(normalize_paper)
    out["dataset_norm"] = out["dataset_raw"].map(normalize_dataset)
    out["metric_norm"] = out["metric_raw"].map(normalize_metric)
    out = out[(out["paper_norm"] != "") & (out["dataset_norm"] != "")]
    out["paper_match_key"] = out["paper_raw"].map(lambda pr: _pwc_identity_from_row(None, str(pr))[0])
    out["paper_match_kind"] = out["paper_raw"].map(lambda pr: _pwc_identity_from_row(None, str(pr))[1])
    return out.drop_duplicates()



In [ ]:
# --- Corpus (pwc_final.json) + inline extraction ---
import random

CORPUS_SELECTION_DF = None
CORPUS_PDF_PATHS: list = []
PIPELINE_TIMING_ROWS: list = []


def _paper_has_metrics(obj: dict) -> bool:
    return bool(_pwc_datasets_list(obj))


def select_corpus_from_pwc_final(
    repo_root: Path,
    *,
    json_basename: str,
    pdf_rel_dir: Path,
    sample_size: int | None,
    sample_seed: int,
    match_threshold: float,
) -> tuple[pd.DataFrame, list]:
    jp = (repo_root / "data" / json_basename).resolve()
    with open(jp, encoding="utf-8") as f:
        papers = json.load(f)
    pdf_dir = (repo_root / pdf_rel_dir).resolve()
    by_arxiv, _by_slug, by_title, by_pdf_stem, by_title_stem = _pwc_final_lookup_maps(papers)
    rows: list[dict] = []
    pdf_paths: list = []
    seen: set[str] = set()
    _n_pdfs = len(list(pdf_dir.glob("*.pdf")))

    for pdf_path in sorted(pdf_dir.glob("*.pdf")):
        stem = pdf_path.stem
        obj, how = _match_pwc_entry_for_pdf_stem(
            stem, by_arxiv, by_title, by_pdf_stem, match_threshold,
            by_title_stem=by_title_stem,
        )
        if obj is None or not _paper_has_metrics(obj):
            continue
        if stem in seen:
            continue
        seen.add(stem)
        pdf_paths.append(pdf_path)
        rows.append({
            "title": str(obj.get("title") or ""),
            "arxiv_id": str(obj.get("arxiv_id") or ""),
            "pdf_stem": stem,
            "pdf_found": True,
            "match_method": how,
            "n_metrics": len(_pwc_datasets_list(obj)),
            "selected": True,
        })

    for obj in papers:
        if not isinstance(obj, dict) or not _paper_has_metrics(obj):
            continue
        stem = ""
        for k in ("local_pdf_path", "local_path"):
            lp = str(obj.get(k) or "").replace("\\", "/")
            if lp:
                stem = Path(lp).stem
                break
        if not stem:
            a = _arxiv_norm_from_text(str(obj.get("arxiv_id") or ""))
            if a:
                stem = a
        if stem in seen:
            continue
        rows.append({
            "title": str(obj.get("title") or ""),
            "arxiv_id": str(obj.get("arxiv_id") or ""),
            "pdf_stem": stem,
            "pdf_found": False,
            "match_method": "",
            "n_metrics": len(_pwc_datasets_list(obj)),
            "selected": False,
        })

    sel_df = pd.DataFrame(rows)
    if sample_size and sample_size > 0 and len(pdf_paths) > sample_size:
        rng = random.Random(sample_seed)
        pdf_paths = sorted(rng.sample(pdf_paths, sample_size), key=lambda p: p.stem)
        picked = {p.stem for p in pdf_paths}
        if len(sel_df):
            sel_df["selected"] = sel_df["pdf_stem"].isin(picked) & sel_df["pdf_found"]
    print(
        f"[corpus] PDFs en carpeta: {_n_pdfs} | "
        f"with JSON+Metrics (for pipeline): {len(pdf_paths)} | "
        f"no match or empty Metrics: {_n_pdfs - len(pdf_paths)}"
    )
    return sel_df, pdf_paths


def build_gt_from_selected_pwc_papers(
    papers_with_pdf: list[dict],
    *,
    skip_empty_datasets: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    dataset_rows, audit_rows = [], []
    for obj in papers_with_pdf:
        title = str(obj.get("title") or "").strip()
        pn = normalize_paper(title)
        dsets = {normalize_dataset(m) for m in _pwc_datasets_list(obj)}
        dsets = {m for m in dsets if m}
        audit_rows.append({
            "title": title, "paper_norm": pn,
            "arxiv_id": str(obj.get("arxiv_id") or ""),
            "n_gt_datasets": len(dsets),
            "gt_datasets_flat": ", ".join(sorted(dsets)),
        })
        if skip_empty_datasets and not dsets or not pn:
            continue
        mk, mk_kind = _dataset_json_identity(obj)
        for d in sorted(dsets):
            dataset_rows.append({
                "paper_raw": title, "dataset_raw": d, "metric_raw": "",
                "paper_norm": pn, "dataset_norm": d,
                "paper_match_key": mk, "paper_match_kind": mk_kind,
            })
    cols = [
        "paper_raw", "dataset_raw", "metric_raw", "paper_norm", "dataset_norm",
        "paper_match_key", "paper_match_kind",
    ]
    gt_df = pd.DataFrame(dataset_rows).drop_duplicates() if dataset_rows else pd.DataFrame(columns=cols)
    return gt_df, pd.DataFrame(audit_rows)


_repo = _find_repo_root_for_extraction()
CORPUS_SELECTION_DF, CORPUS_PDF_PATHS = select_corpus_from_pwc_final(
    _repo,
    json_basename=PWC_ORIGIN_JSON_BASENAME,
    pdf_rel_dir=PDF_FILES_DIR,
    sample_size=PWC_SAMPLE_SIZE,
    sample_seed=PWC_SAMPLE_SEED,
    match_threshold=MATCH_THRESHOLD,
)
print(f"[corpus] PDFs for pipeline: {len(CORPUS_PDF_PATHS)}")

print(f"[corpus] Next: run the «Extraction (single code cell)» cell.")




## Extraction (single code cell)

Run the next code cell after corpus selection (`CORPUS_PDF_PATHS`).

It runs LightOnOCR caches, GLiNER, builds Tables With Datasets, writes the combination Excel, and sets `INPUT_EXCEL` plus `PIPELINE_TIMING_ROWS`.


In [ ]:
# --- Extraction: LightOnOCR + GLiNER + Excel ---
# Requires CORPUS_PDF_PATHS from the corpus selection cell.


# ========================================================================
# [merged from former cell 14]
# ========================================================================

import json
import os
import re
import time
import tempfile
import warnings
from pathlib import Path

import pandas as pd
import torch
from bs4 import BeautifulSoup

# ─── Section 2a — Optional installation of OCR / GLiNER2 dependencies ────────
import sys, subprocess

AUTO_INSTALL_DEPS = False

INSTALL_SPECS = [
    "gliner2>=1.2.5",
    "transformers @ git+https://github.com/huggingface/transformers.git",
    "pypdfium2",
    "pillow",
    "accelerate",
    "beautifulsoup4",
    "openpyxl",
    # Needed for torch 2.11+cu130 to JIT-compile fused kernels (DeBERTa-v3 positional
    # buckets). Without this you get: "nvrtc: failed to open libnvrtc-builtins.so.13.0".
    "nvidia-cuda-nvrtc-cu13",
]

if False and AUTO_INSTALL_DEPS:
    cmd = [sys.executable, "-m", "pip", "install", "-q", "-U", *INSTALL_SPECS]
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)
    print("If transformers was updated you may need to restart the kernel.")


# ========================================================================
# [merged from former cell 15]
# ========================================================================

# ─── Section 2c — Load LightOnOCR model ──────────────────────────────────────
import pypdfium2 as pdfium
from PIL import Image
from transformers import LightOnOcrForConditionalGeneration, LightOnOcrProcessor

OCR_MODEL_ID = "lightonai/LightOnOCR-2-1B"
OCR_TARGET_LONGEST = 1540  # px, per LightOnOCR model card
OCR_MAX_NEW_TOKENS = 8192

if torch.cuda.is_available():
    ocr_device = "cuda"
    ocr_dtype = torch.bfloat16
elif torch.backends.mps.is_available():
    ocr_device = "mps"
    ocr_dtype = torch.float32
else:
    ocr_device = "cpu"
    ocr_dtype = torch.float32

print(f"OCR device: {ocr_device}, dtype: {ocr_dtype}")

ocr_processor = LightOnOcrProcessor.from_pretrained(OCR_MODEL_ID)
ocr_model = LightOnOcrForConditionalGeneration.from_pretrained(
    OCR_MODEL_ID,
    torch_dtype=ocr_dtype,
    attn_implementation="eager",
).to(ocr_device)

print(f"OCR model loaded : {OCR_MODEL_ID}")


# ========================================================================
# [merged from former cell 16]
# ========================================================================

# ─── Section 2d — OCR helpers ────────────────────────────────────────────────
import tempfile, os


def render_pdf_page(pdf_doc, page_idx: int, target_longest: int = OCR_TARGET_LONGEST) -> Image.Image:
    """Render a PDF page to PIL RGB image at 200 DPI, max `target_longest` px on the longest side."""
    page = pdf_doc[page_idx]
    bitmap = page.render(scale=200 / 72)  # 200 DPI
    pil_image = bitmap.to_pil()

    w, h = pil_image.size
    longest = max(w, h)
    if longest > target_longest:
        ratio = target_longest / longest
        pil_image = pil_image.resize((int(w * ratio), int(h * ratio)), Image.LANCZOS)

    if pil_image.mode != "RGB":
        pil_image = pil_image.convert("RGB")
    return pil_image


def ocr_page(pil_image: Image.Image, max_new_tokens: int = OCR_MAX_NEW_TOKENS) -> str:
    """Run LightOnOCR on a page image. Returns the raw generated text (markdown/HTML)."""
    tmp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
    pil_image.save(tmp, format="PNG")
    tmp.close()
    try:
        conversation = [
            {"role": "user", "content": [{"type": "image", "url": tmp.name}]}
        ]
        inputs = ocr_processor.apply_chat_template(
            conversation,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        )
        inputs = {
            k: v.to(device=ocr_device, dtype=ocr_dtype) if v.is_floating_point() else v.to(ocr_device)
            for k, v in inputs.items()
        }
        with torch.no_grad():
            output_ids = ocr_model.generate(**inputs, max_new_tokens=max_new_tokens)
        generated_ids = output_ids[0, inputs["input_ids"].shape[1]:]
        return ocr_processor.decode(generated_ids, skip_special_tokens=True)
    finally:
        os.unlink(tmp.name)


def _extract_html_tables(text: str) -> list[str]:
    """Return every <table>…</table> block present in the OCR output."""
    return re.findall(r"<table\b[^>]*>.*?</table>", text, flags=re.DOTALL | re.IGNORECASE)


print("OCR helpers ready.")


# ========================================================================
# [merged from former cell 17]
# ========================================================================

# ─── Section 2e — LightOnOCR extraction (with JSON cache) ────────────────────

def _pdf_readable(path_pdf: Path) -> tuple[bool, str]:
    """Quick checks before PDFium (corrupt downloads often fail with Data format error)."""
    path_pdf = Path(path_pdf)
    if not path_pdf.is_file():
        return False, "file missing"
    try:
        if path_pdf.stat().st_size < 128:
            return False, "file too small"
        with open(path_pdf, "rb") as f:
            if f.read(5) != b"%PDF-":
                return False, "missing %PDF- header"
    except OSError as e:
        return False, str(e)
    return True, ""


def run_lightonocr(path_pdf: Path, cache_dir: Path, verbose: bool = False) -> dict:
    """Extract tables from one PDF using LightOnOCR. Cached as <stem>_lightonocr.json."""
    path_pdf = Path(path_pdf)
    cache_file = cache_dir / f"{path_pdf.stem}_lightonocr.json"

    if cache_file.exists():
        with open(cache_file, "r", encoding="utf-8") as f:
            result = json.load(f)
        if verbose:
            n = sum(len(p["tables"]) for p in result["results"])
            print(f"[CACHE] {path_pdf.name} ({n} tables)")
        return result

    ok, why = _pdf_readable(path_pdf)
    if not ok:
        raise ValueError(f"unreadable PDF ({why})")

    if verbose:
        print(f"[RUN] {path_pdf.name}")

    try:
        pdf_doc = pdfium.PdfDocument(str(path_pdf))
    except Exception as e:
        raise RuntimeError(f"Pdfium cannot open {path_pdf.name}: {e}") from e

    results_data = []
    try:
        for page_idx in range(len(pdf_doc)):
            page_num = page_idx + 1
            pil_image = render_pdf_page(pdf_doc, page_idx)
            ocr_text = ocr_page(pil_image)
            html_tables = _extract_html_tables(ocr_text)
            if html_tables:
                results_data.append({
                    "page": page_num,
                    "tables": [{"html": t} for t in html_tables],
                })
    finally:
        pdf_doc.close()

    result = {"file_name": str(path_pdf), "results": results_data}
    with open(cache_file, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    return result


ocr_outputs: dict = {}
extraction_rows = []
ocr_skipped_rows: list[dict] = []
import time as _time_ocr
_t_ocr_start = _time_ocr.perf_counter()

PDF_DIR = _resolve_pdf_files_dir(_find_repo_root_for_extraction())
cache_files = sorted(PDF_DIR.glob("*_lightonocr.json"))
to_process: list[tuple[str, Path | None]] = []

if not CORPUS_PDF_PATHS:
    raise RuntimeError(
        "CORPUS_PDF_PATHS is empty. Run the corpus selection cell first."
    )

if CORPUS_PDF_PATHS:
    to_process = [(p.stem, p) for p in CORPUS_PDF_PATHS]
elif cache_files:
    print(
        f"⚠  No PDFs found in {PDF_DIR.resolve()}, but {len(cache_files)} "
        f"*_lightonocr.json caches are present. Using caches directly "
        f"(LightOnOCR cannot be re-run without the source PDFs)."
    )
    to_process = [
        (cache.name.replace("_lightonocr.json", ""), None)
        for cache in cache_files
    ]
else:
    raise FileNotFoundError(
        f"No PDFs and no *_lightonocr.json caches found in "
        f"{PDF_DIR.resolve()}. Restore either the PDFs or the JSON caches. "
        f"(CWD = {Path.cwd()})."
    )

n_total = len(to_process)
for i, (stem, pdf) in enumerate(to_process, start=1):
    cache_file = PDF_DIR / f"{stem}_lightonocr.json"
    if pdf is not None:
        source = "cache" if cache_file.exists() else "run"
        print(f"[{i}/{n_total}] {stem} ({source})")
        try:
            ocr_outputs[stem] = run_lightonocr(pdf, cache_dir=PDF_DIR, verbose=SHOW_VERBOSE)
        except Exception as e:
            print(f"  [SKIP] {stem}: {e}")
            ocr_skipped_rows.append({"pdf_stem": stem, "pdf_path": str(pdf), "error": str(e)})
            extraction_rows.append({"paper": stem, "source": "pdf_error", "tables": 0})
            continue
    else:
        source = "cache-only"
        print(f"[{i}/{n_total}] {stem} (cache-only)")
        try:
            with open(cache_file, encoding="utf-8") as f:
                ocr_outputs[stem] = json.load(f)
        except Exception as e:
            print(f"  [SKIP] {stem}: bad cache ({e})")
            ocr_skipped_rows.append({"pdf_stem": stem, "pdf_path": "", "error": f"cache: {e}"})
            extraction_rows.append({"paper": stem, "source": "cache_error", "tables": 0})
            continue
    n_tables = sum(len(p["tables"]) for p in ocr_outputs[stem]["results"])
    extraction_rows.append({"paper": stem, "source": source, "tables": n_tables})

extract_df = pd.DataFrame(
    extraction_rows,
    columns=["paper", "source", "tables"],
).sort_values(["tables", "paper"], ascending=[False, True])
_ocr_seconds = _time_ocr.perf_counter() - _t_ocr_start
if ocr_skipped_rows:
    print(f"OCR skipped (unreadable PDF / error): {len(ocr_skipped_rows)}")
    for _r in ocr_skipped_rows[:10]:
        print(f"  - {_r['pdf_stem']}: {_r['error']}")
    if len(ocr_skipped_rows) > 10:
        print(f"  … and {len(ocr_skipped_rows) - 10} more")
print(f"OCR phase: {len(ocr_outputs)} ok / {len(to_process)} total, {_ocr_seconds:.1f}s")


# ========================================================================
# [merged from former cell 18]
# ========================================================================

# ─── Section 3a — Disable PyTorch JIT / TensorExpr fusion ────────────────────
# DeBERTa-v3 (the encoder used by gliner2) triggers a TensorExpr fusion on the
# relative-position-bucket path, which calls nvrtc at runtime. On some servers
# (e.g. torch 2.11+cu130 without `nvidia-cuda-nvrtc-cu13`) this fails with:
#   RuntimeError: nvrtc: error: failed to open libnvrtc-builtins.so.13.0
# Disabling the fusers forces eager kernels (tiny slowdown, fully correct).
import os
os.environ.setdefault("PYTORCH_JIT", "0")
os.environ.setdefault("PYTORCH_NVFUSER_DISABLE", "1")

import torch
for name, args in [
    ("_jit_set_profiling_executor", (False,)),
    ("_jit_set_profiling_mode",     (False,)),
    ("_jit_override_can_fuse_on_gpu", (False,)),
    ("_jit_override_can_fuse_on_cpu", (False,)),
    ("_jit_set_texpr_fuser_enabled", (False,)),
    ("_jit_set_nvfuser_enabled",     (False,)),
]:
    fn = getattr(torch._C, name, None)
    if fn is not None:
        try:
            fn(*args)
        except Exception as _e:
            print(f"  (skipped torch._C.{name}: {_e})")

print("JIT / TensorExpr / nvFuser fusers disabled.")


# ========================================================================
# [merged from former cell 19]
# ========================================================================

# ─── Section 3 — GLiNER 2.0 model ────────────────────────────────────────────
from gliner2 import GLiNER2
import gliner2 as _gliner2

GLINER2_MODEL_ID = "fastino/gliner2-base-v1"   # base has better recall on KGE tables than large

gliner2_map_location = "cuda" if torch.cuda.is_available() else "cpu"

gliner2_model = GLiNER2.from_pretrained(
    GLINER2_MODEL_ID,
    map_location=gliner2_map_location,
)

print(f"GLiNER2 available: {getattr(_gliner2, '__version__', 'unknown')}")
print(f"GLiNER2 model    : {GLINER2_MODEL_ID} on {gliner2_map_location}")


# ========================================================================
# [merged from former cell 20]
# ========================================================================

# ─── Section 4 — Pure GLiNER 2.0 combination extraction ──────────────────────────

GLINER2_LABEL_DESCRIPTIONS = {
    "model": "Name of a machine-learning model, knowledge-graph embedding method, algorithm or system (e.g. TransE, ComplEx, HolE, RotatE).",
    "dataset": "Name of a benchmark dataset or knowledge-graph corpus (e.g. WN18, WN18RR, FB15k, FB15k-237, YAGO, NELL).",
    "metric": "Name of an evaluation metric used to score a model (e.g. MRR, Hits@1, Hits@3, Hits@10, MR, Accuracy, F1).",
}
GLINER2_LABELS = list(GLINER2_LABEL_DESCRIPTIONS.keys())
GLINER2_MIN_SCORE = 0.65            # raise threshold to cut noisy cross-labeled entities
GLINER2_MAX_CHARS = 3000
GLINER2_USE_CONFIDENCE = True      # verified working after the JIT fix
GLINER2_DEBUG_ERRORS = True        # print the first N extraction errors instead of swallowing them
_gliner2_error_count = {"n": 0}
_GLINER2_ERROR_LIMIT = 3

# Raw GLiNER output — blacklist disabled in minimal pipeline.
USE_BLACKLIST = False
GLINER2_LABEL_BLACKLIST: dict[str, set[str]] = {"model": set(), "dataset": set(), "metric": set()}


# ── Helpers ──────────────────────────────────────────────────────────────────

def _rows_from_html(html: str) -> list[str]:
    """One text string per <tr>, cells joined by ' | '."""
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    for tr in soup.find_all("tr"):
        cells = [c.get_text(separator=" ", strip=True) for c in tr.find_all(["th", "td"])]
        cells = [c for c in cells if c]
        if cells:
            rows.append(" | ".join(cells))
    return rows


def _header_and_rows_from_html(html: str) -> tuple[list[str], list[str]]:
    """Split an HTML table into (header_lines, body_lines).

    A row is treated as header if it lives inside <thead> OR if every cell is
    a <th>. If no header is detected, the first row is used as header (common
    in OCR output that omits <thead>).
    """
    soup = BeautifulSoup(html, "html.parser")
    thead = soup.find("thead")
    tbody = soup.find("tbody")

    def _row_text(tr):
        cells = [c.get_text(separator=" ", strip=True) for c in tr.find_all(["th", "td"])]
        cells = [c for c in cells if c]
        return " | ".join(cells) if cells else None

    header_lines: list[str] = []
    body_lines: list[str] = []

    if thead is not None:
        for tr in thead.find_all("tr"):
            txt = _row_text(tr)
            if txt:
                header_lines.append(txt)

    trs = tbody.find_all("tr") if tbody is not None else soup.find_all("tr")
    for i, tr in enumerate(trs):
        # skip rows already consumed by thead
        if thead is not None and tr in thead.find_all("tr"):
            continue
        cells = tr.find_all(["th", "td"])
        if not cells:
            continue
        txt = _row_text(tr)
        if not txt:
            continue
        all_th = all(c.name == "th" for c in cells)
        if all_th and not body_lines:
            header_lines.append(txt)
        else:
            body_lines.append(txt)

    # Fallback: if we still have no header but do have body rows, use the first
    # body row as the header (common in LightOnOCR output without <thead>).
    if not header_lines and body_lines:
        header_lines = [body_lines[0]]
        body_lines = body_lines[1:]

    return header_lines, body_lines


def _clean_entity(value: str) -> str:
    """Light cleanup: strip citation markers, LaTeX math mode and whitespace."""
    s = str(value).strip()
    s = re.sub(r"\[[^\]]{1,50}\]", "", s)                                   # [1], [Smith 2020]
    s = re.sub(r"\((?:[^\)]*\d{4}[^\)]*|[^\)]*et\s*al\.?[^\)]*)\)", "", s, flags=re.IGNORECASE)
    # LaTeX math mode: $...$  →  content inside
    s = re.sub(r"\$([^$]+?)\$", r"\1", s)
    # Common LaTeX wrappers (must run BEFORE stripping braces)
    s = re.sub(r"\\(?:textbf|textit|text|mathbf|mathrm|mathit)\{([^{}]*)\}", r"\1", s)
    # Sub/super-scripts: _{xxx}, ^{xxx}  →  xxx
    for _ in range(3):
        s = re.sub(r"[_^]\{([^{}]*)\}", r"\1", s)
    # Remaining stray braces
    s = s.replace("{", "").replace("}", "")
    # Tighten "WD ++" / "WD --" that came from "WD $_{++}$" etc.
    s = re.sub(r"(\w)\s+(\+\+|--)(?=\s|$)", r"\1\2", s)
    s = re.sub(r"\s+", " ", s).strip(" .,:;-")
    return s


def _extract_entities_raw(text: str) -> dict[str, tuple[str, float]]:
    """Return the raw best-label-per-text mapping (pre-dedup to the label dict).

    Returned shape: {entity_text: (label, confidence)}. The caller can then
    aggregate across rows of the same table before assigning final labels.
    """
    if not text.strip():
        return {}
    try:
        if GLINER2_USE_CONFIDENCE:
            result = gliner2_model.extract_entities(
                text[:GLINER2_MAX_CHARS],
                GLINER2_LABEL_DESCRIPTIONS,
                include_confidence=True,
            )
        else:
            result = gliner2_model.extract_entities(
                text[:GLINER2_MAX_CHARS],
                GLINER2_LABEL_DESCRIPTIONS,
            )
    except Exception as e:
        if GLINER2_DEBUG_ERRORS and _gliner2_error_count["n"] < _GLINER2_ERROR_LIMIT:
            _gliner2_error_count["n"] += 1
            print(f"[gliner2 error #{_gliner2_error_count['n']}] {type(e).__name__}: {e}")
            print(f"  on text: {text[:200]!r}")
        return {}

    ents_by_label = (result or {}).get("entities", {}) or {}
    best: dict[str, tuple[str, float]] = {}
    for label, items in ents_by_label.items():
        if label not in GLINER2_LABELS:
            continue
        for item in items or []:
            if isinstance(item, dict):
                raw_text = str(item.get("text", ""))
                score = float(item.get("confidence", 1.0) or 1.0)
            else:
                raw_text = str(item)
                score = 1.0
            value = _clean_entity(raw_text)
            if score < GLINER2_MIN_SCORE or not value or len(value) < 2:
                continue
            if re.fullmatch(r"[+-]?\d+(?:\.\d+)?", value):
                continue
            if USE_BLACKLIST and value.lower() in GLINER2_LABEL_BLACKLIST.get(label, set()):
                continue
            prev = best.get(value)
            if prev is None or score > prev[1]:
                best[value] = (label, score)
    return best


def _extract_entities(text: str) -> dict[str, list[str]]:
    """Run GLiNER 2.0 on a text fragment and return detected entities per label
    with per-call dedup (each entity text lands under its highest-confidence label)."""
    best = _extract_entities_raw(text)
    found: dict[str, set[str]] = {label: set() for label in GLINER2_LABELS}
    for value, (label, _score) in best.items():
        found[label].add(value)
    return {k: sorted(v) for k, v in found.items()}


def _merge_best(target: dict[str, tuple[str, float]], other: dict[str, tuple[str, float]]) -> None:
    """Merge `other` into `target`, keeping the highest-confidence label per text."""
    for value, (label, score) in other.items():
        prev = target.get(value)
        if prev is None or score > prev[1]:
            target[value] = (label, score)


def _labels_from_best(best: dict[str, tuple[str, float]]) -> dict[str, list[str]]:
    found: dict[str, set[str]] = {label: set() for label in GLINER2_LABELS}
    for value, (label, _score) in best.items():
        found[label].add(value)
    return {k: sorted(v) for k, v in found.items()}


def _combinations_from_entities(ents: dict[str, list[str]]) -> list[tuple[str, str, str]]:
    return [(m, d, mt) for m in ents["model"] for d in ents["dataset"] for mt in ents["metric"]]


import time as _time_gliner
_t_gliner_start = _time_gliner.perf_counter()



# ── Main extraction loop ────────────────────────────────────────────────────

combination_rows: list[dict] = []
table_meta_rows: list[dict] = []

for pdf_stem, ocr_result in ocr_outputs.items():
    for page_data in ocr_result.get("results", []):
        page_num = int(page_data.get("page", 0) or 0)
        for table_idx, table in enumerate(page_data.get("tables", []), start=1):
            table_name = f"{pdf_stem}_p{page_num}_t{table_idx}"
            html = table.get("html", "")
            if not html:
                continue

            header_lines, body_lines = _header_and_rows_from_html(html)
            if not header_lines and not body_lines:
                continue

            header_context = "\n".join(header_lines)

            # --- Pass 1: header alone (anchors dataset/metric labels strongly) ---
            table_best: dict[str, tuple[str, float]] = {}
            if header_context:
                _merge_best(table_best, _extract_entities_raw(header_context))

            # --- Pass 2: each body row, prefixed with the header for context ---
            for row_text in body_lines:
                if header_context:
                    prompt = (
                        f"Table column headers: {header_context}\n"
                        f"Row: {row_text}"
                    )
                else:
                    prompt = row_text
                _merge_best(table_best, _extract_entities_raw(prompt))

            # --- Fallback: feed the whole table as a single chunk if nothing found
            if not table_best:
                full = "\n".join(header_lines + body_lines)
                _merge_best(table_best, _extract_entities_raw(full))

            # After both passes, resolve each unique text to a single label (the
            # one with highest confidence across all detections). This prevents
            # "LMF" from leaking into `dataset` when it's really a `model`.
            ents = _labels_from_best(table_best)
            all_models = set(ents["model"])
            all_datasets = set(ents["dataset"])
            all_metrics = set(ents["metric"])

            # --- Form combinations: model × dataset × metric at table level ---
            table_combinations: set[tuple[str, str, str]] = set(_combinations_from_entities(ents))
            for d in all_datasets:
                if not any(normalize_dataset(t[1]) == d for t in table_combinations if t[1]):
                    for m in (sorted(all_models) or [""]):
                        for mt in (sorted(all_metrics) or [""]):
                            table_combinations.add((m, d, mt))

            table_meta_rows.append({
                "paper": pdf_stem,
                "table_name": table_name,
                "models": " | ".join(sorted(all_models)) or "(none)",
                "datasets": " | ".join(sorted(all_datasets)) or "(none)",
                "metrics": " | ".join(sorted(all_metrics)) or "(none)",
                "combinations_found": len(table_combinations),
            })

            for m, d, mt in sorted(table_combinations):
                combination_rows.append({
                    "paper": pdf_stem,
                    "table_name": table_name,
                    "model": m,
                    "dataset": d,
                    "metric": mt,
                })

# ── Build dataframes ────────────────────────────────────────────────────────

combinations_df = pd.DataFrame(combination_rows) if combination_rows else pd.DataFrame(
    columns=["paper", "table_name", "model", "dataset", "metric"]
)
combinations_df = combinations_df.drop_duplicates().sort_values(
    ["paper", "table_name", "model", "dataset", "metric"]
).reset_index(drop=True)

table_meta_df = pd.DataFrame(table_meta_rows)

print("=" * 70)
print("PURE GLINER 2.0 COMBINATION EXTRACTION (LightOnOCR tables)")
print("=" * 70)
print(f"Blacklist mode : {'ON (post-filter applied)' if USE_BLACKLIST else 'OFF (raw GLiNER2 output)'}")
print(f"Total combinations : {len(combinations_df)}")
print(f"Unique models  : {combinations_df['model'].nunique() if len(combinations_df) else 0}")
print(f"Unique datasets: {combinations_df['dataset'].nunique() if len(combinations_df) else 0}")
print(f"Unique metrics : {combinations_df['metric'].nunique() if len(combinations_df) else 0}")
print()
display(combinations_df)
print()
gliner_seconds = _time_gliner.perf_counter() - _t_gliner_start
print(f"GLiNER phase: {gliner_seconds:.1f}s")
print("Table metadata:")
display(table_meta_df)


# ========================================================================
# [merged from former cell 21]
# ========================================================================

# ─── Tables With Datasets (dataset benchmark: prioritize dataset column)

dataset_table_rows = []

tables_with_datasets_meta = table_meta_df[
    (table_meta_df["datasets"] != "(none)") | (table_meta_df["metrics"] != "(none)")
].copy()

for _, meta in tables_with_datasets_meta.iterrows():
    paper = meta["paper"]
    tname = meta["table_name"]

    table_trips = combinations_df[
        (combinations_df["paper"] == paper) & (combinations_df["table_name"] == tname)
    ]

    if not table_trips.empty:
        for _, trip in table_trips.iterrows():
            ds = str(trip.get("dataset", "") or "").strip()
            if not ds:
                continue
            dataset_table_rows.append({
                "paper": paper,
                "table_name": tname,
                "model": trip.get("model", ""),
                "dataset": ds,
                "metric": trip.get("metric", ""),
                "has_combination": True,
            })
    else:
        datasets = meta["datasets"] if meta["datasets"] != "(none)" else ""
        dataset_list = [d.strip() for d in str(datasets).split(" | ") if d.strip()]
        for dataset in dataset_list:
            dataset_table_rows.append({
                "paper": paper,
                "table_name": tname,
                "model": "",
                "dataset": dataset,
                "metric": "",
                "has_combination": False,
            })

tables_with_metrics_df = pd.DataFrame(dataset_table_rows)
if not tables_with_metrics_df.empty:
    tables_with_metrics_df = tables_with_metrics_df.drop_duplicates().sort_values(
        ["paper", "table_name", "model", "dataset", "metric"]
    ).reset_index(drop=True)

n_with = int(tables_with_metrics_df["has_combination"].sum()) if len(tables_with_metrics_df) else 0
n_without = len(tables_with_metrics_df) - n_with

print(f"Tables with datasets (meta rows): {len(tables_with_datasets_meta)}")
print(f"  Rows with full combination    : {n_with}")
print(f"  Rows dataset-only (partial)   : {n_without}")
print()
display(tables_with_metrics_df)


# ========================================================================
# [merged from former cell 22]
# ========================================================================

# ─── Tables With Values: same detail as "Tables With Datasets" + numeric value
# Looks up, for every (paper, table_name, model, dataset, metric) row, the cell
# value at the intersection of the model row and the column whose header text
# matches the dataset + metric. The output mirrors `tables_with_metrics_df`
# but adds a `value` column.

_NUMERIC_RE = re.compile(r"[-+]?\d+(?:[.,]\d+)?")


def _parse_table_grid(
    html: str,
) -> tuple[list[list[str]], list[bool], list[list[bool]], int]:
    """Parse HTML table into a 2D text grid, expanding colspan/rowspan.

    Returns (grid, is_header_mask, fixed_mask, max_cols) where:
      * `grid[r][c]` is the cell text after expansion (rows may be ragged —
        padding is left to the caller / rebalance step).
      * `is_header_mask[r]` is True iff every cell in row r is a <th>.
      * `fixed_mask[r][c]` is True iff the cell was declared with rowspan>1
        or was inherited from a rowspan in a previous row. These cells must
        NOT be shifted by the rebalance step.
      * `max_cols` is the widest row (usually the body / leaf-header width).
    """
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table") or soup
    tr_list = table.find_all("tr")

    grid: list[list[str]] = []
    is_header: list[bool] = []
    fixed: list[list[bool]] = []
    pending: dict[int, tuple[str, int]] = {}  # col_idx -> (text, rows_left)

    def _ensure(row, fmask, n):
        while len(row) <= n:
            row.append("")
            fmask.append(False)

    for tr in tr_list:
        row: list[str] = []
        fmask: list[bool] = []
        cells = tr.find_all(["td", "th"])
        is_header.append(bool(cells) and all(c.name == "th" for c in cells))

        c_idx = 0
        for cell in cells:
            while pending.get(c_idx, (None, 0))[1] > 0:
                text, rem = pending[c_idx]
                _ensure(row, fmask, c_idx)
                row[c_idx] = text
                fmask[c_idx] = True  # inherited from rowspan
                pending[c_idx] = (text, rem - 1)
                c_idx += 1

            text = re.sub(r"\s+", " ", cell.get_text(separator=" ", strip=True))
            colspan = int(cell.get("colspan", 1) or 1)
            rowspan = int(cell.get("rowspan", 1) or 1)
            is_fixed = rowspan > 1
            for _ in range(colspan):
                _ensure(row, fmask, c_idx)
                row[c_idx] = text
                fmask[c_idx] = is_fixed
                if is_fixed:
                    pending[c_idx] = (text, rowspan - 1)
                c_idx += 1

        for col in sorted(pending):
            if pending[col][1] > 0 and col >= len(row):
                text, rem = pending[col]
                while len(row) < col:
                    row.append("")
                    fmask.append(False)
                row.append(text)
                fmask.append(True)
                pending[col] = (text, rem - 1)

        grid.append(row)
        fixed.append(fmask)

    max_cols = max((len(r) for r in grid), default=0)
    return grid, is_header, fixed, max_cols


def _rebalance_header_grid(
    grid: list[list[str]],
    is_header: list[bool],
    fixed: list[list[bool]],
    max_cols: int,
) -> list[list[str]]:
    """Fix header rows that are narrower than the body, redistributing
    non-fixed (non-rowspan) cells evenly across the body width.

    This is needed because LightOnOCR sometimes emits outer header cells with
    colspan that doesn't match the leaf-row width (e.g. `<th colspan="2">WN18`
    when the leaf row actually has 4 sub-columns under WN18). Without this
    fix, column signatures would be misaligned and (dataset, metric) lookups
    would return the wrong values.

    Fixed cells (rowspan > 1, or inherited from a rowspan) keep their
    original column index. Non-fixed cells are spread uniformly across the
    remaining columns. Rows whose width already matches `max_cols`, or where
    the mismatch cannot be resolved with an integer factor, are just padded.
    """
    new_grid: list[list[str]] = []
    for r_idx, row in enumerate(grid):
        cur_w = len(row)
        if not is_header[r_idx] or cur_w == max_cols:
            padded = list(row) + [""] * (max_cols - cur_w)
            new_grid.append(padded)
            continue

        fmask = fixed[r_idx]
        explicit_idx = [i for i, f in enumerate(fmask) if not f]
        fixed_idx = [i for i, f in enumerate(fmask) if f]
        n_exp = len(explicit_idx)
        n_fix = len(fixed_idx)

        if n_exp == 0:
            new_grid.append(list(row) + [""] * (max_cols - cur_w))
            continue

        target = max_cols - n_fix
        factor, rem = divmod(target, n_exp)
        if factor <= 1 or rem != 0:
            # Can't rebalance cleanly — fall back to padding
            new_grid.append(list(row) + [""] * (max_cols - cur_w))
            continue

        new_row = [""] * max_cols
        # Place fixed cells at their original column indices
        fixed_cols_set = set()
        for i in fixed_idx:
            if i < max_cols:
                new_row[i] = row[i]
                fixed_cols_set.add(i)
        # Remaining column slots, in order, receive explicit cells (each
        # cell expanded to `factor` contiguous columns).
        free = [c for c in range(max_cols) if c not in fixed_cols_set]
        if len(free) != target:
            new_grid.append(list(row) + [""] * (max_cols - cur_w))
            continue
        pos = 0
        for i in explicit_idx:
            for _ in range(factor):
                new_row[free[pos]] = row[i]
                pos += 1
        new_grid.append(new_row)

    return new_grid


def _looks_numeric(text: str) -> bool:
    if not text:
        return False
    s = text.replace(",", "").replace("%", "").replace("$", "")
    s = s.replace("\\", "").replace("_", "").replace("^", "").replace("±", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return bool(s) and any(_NUMERIC_RE.fullmatch(tok) for tok in s.split())


def _norm_for_match(s: str) -> str:
    s = (s or "").lower().replace("(", " ").replace(")", " ").replace("%", " ")
    return re.sub(r"\s+", " ", s).strip()


def _values_from_table(
    html: str, model: str, dataset: str, metric: str,
) -> list[tuple[str, str]]:
    """Return all (variant, value) pairs at (row=model, col≈dataset+metric).

    When a (model, dataset, metric) cell is subdivided by a third header level
    (e.g. 'raw' / 'filter' under each metric), each sub-column becomes one
    (variant, value) pair where `variant` is the leaf header text of that
    column. If there is no sub-variant (leaf equals the metric or dataset
    itself), `variant` is "". Matching is case-insensitive and tolerant of
    whitespace, parentheses and percent signs so 'Hits@10(%)' cuadra con
    'Hits@10 (%)' y 'FB15K' con 'FB15k'. Returns [] when nothing matches.
    """
    if not html or not (model or metric):
        return []
    try:
        grid, is_header, fixed, max_cols = _parse_table_grid(html)
        grid = _rebalance_header_grid(grid, is_header, fixed, max_cols)
    except Exception:
        return []
    if not grid or not any(is_header):
        return []

    last_header = max(i for i, h in enumerate(is_header) if h)
    n_cols = max_cols

    # For each column, build the full header path (list of distinct texts top→bottom)
    col_path: list[list[str]] = []
    for c in range(n_cols):
        path: list[str] = []
        for r in range(last_header + 1):
            cell = grid[r][c] if c < len(grid[r]) else ""
            if cell and (not path or path[-1] != cell):
                path.append(cell)
        col_path.append(path)
    col_sig = [_norm_for_match(" ".join(p)) for p in col_path]
    col_leaf = [p[-1] if p else "" for p in col_path]

    m_norm = _norm_for_match(metric)
    d_norm = _norm_for_match(dataset)

    # Prefer columns whose signature contains BOTH dataset and metric
    cand_cols = [
        c for c, key in enumerate(col_sig)
        if (not d_norm or d_norm in key) and (not m_norm or m_norm in key)
    ]
    # Fallback: match metric only (tables with a single dataset)
    if not cand_cols and m_norm:
        cand_cols = [c for c, key in enumerate(col_sig) if m_norm in key]
    if not cand_cols:
        return []

    def _variant_for(col_idx: int) -> str:
        """Return the leaf header of this column, or '' if it just echoes
        the metric / dataset (i.e. the column has no sub-variant)."""
        leaf = col_leaf[col_idx].strip()
        leaf_norm = _norm_for_match(leaf)
        if not leaf_norm:
            return ""
        if m_norm and (leaf_norm == m_norm or leaf_norm in m_norm or m_norm in leaf_norm):
            return ""
        if d_norm and (leaf_norm == d_norm or leaf_norm in d_norm or d_norm in leaf_norm):
            return ""
        return leaf

    model_norm = _norm_for_match(model)
    for r in range(last_header + 1, len(grid)):
        row = grid[r]
        if not any(_looks_numeric(x) for x in row):
            continue
        # Leading non-numeric cells form the row label
        leading: list[str] = []
        for cell in row:
            if _looks_numeric(cell):
                break
            leading.append(cell)
        lead_norm = _norm_for_match(" ".join(leading))
        if not (model_norm and model_norm in lead_norm):
            continue
        # Collect one (variant, value) per candidate column in this row
        results: list[tuple[str, str]] = []
        seen: set[tuple[str, str]] = set()
        # Prefer numeric cells; fall back to any non-empty cell
        for c in cand_cols:
            if c >= len(row):
                continue
            val = row[c].strip()
            if not val or not _looks_numeric(val):
                continue
            variant = _variant_for(c)
            key = (variant, val)
            if key in seen:
                continue
            seen.add(key)
            results.append((variant, val))
        if not results:
            for c in cand_cols:
                if c < len(row) and row[c].strip():
                    variant = _variant_for(c)
                    key = (variant, row[c].strip())
                    if key in seen:
                        continue
                    seen.add(key)
                    results.append((variant, row[c].strip()))
        if results:
            return results
    return []


# Index: table_name -> raw HTML (once)
_html_by_table: dict[str, str] = {}
for _stem, _res in ocr_outputs.items():
    for _pd in _res.get("results", []):
        _pn = int(_pd.get("page", 0) or 0)
        for _ti, _tb in enumerate(_pd.get("tables", []), start=1):
            _html_by_table[f"{_stem}_p{_pn}_t{_ti}"] = _tb.get("html", "")


_tw_values_rows: list[dict] = []
for _, r in tables_with_metrics_df.iterrows():
    html = _html_by_table.get(r["table_name"], "")
    pairs = _values_from_table(html, r["model"], r["dataset"], r["metric"])
    if not pairs:
        # Keep one row so the model/dataset/metric is still represented,
        # even if we couldn't resolve a numeric value.
        _tw_values_rows.append({
            "paper": r["paper"],
            "table_name": r["table_name"],
            "model": r["model"],
            "dataset": r["dataset"],
            "metric": r["metric"],
            "variant": "",
            "value": "",
            "has_combination": bool(r["has_combination"]),
        })
        continue
    for variant, val in pairs:
        _tw_values_rows.append({
            "paper": r["paper"],
            "table_name": r["table_name"],
            "model": r["model"],
            "dataset": r["dataset"],
            "metric": r["metric"],
            "variant": variant,
            "value": val,
            "has_combination": bool(r["has_combination"]),
        })

tables_with_values_df = pd.DataFrame(_tw_values_rows)

_n_val = int((tables_with_values_df["value"] != "").sum())
_n_variant = int((tables_with_values_df["variant"] != "").sum())
print(f"Tables With Values: {len(tables_with_values_df)} rows")
print(f"  Rows with numeric value : {_n_val}")
print(f"  Rows without value      : {len(tables_with_values_df) - _n_val}")
print(f"  Rows with sub-variant   : {_n_variant}  (e.g. 'raw' / 'filter')")
display(tables_with_values_df.head(25))


# ========================================================================
# [merged from former cell 23]
# ========================================================================

# ─── Section 5 — Export to Excel ─────────────────────────────────────────────
from openpyxl.styles import Font, PatternFill
from openpyxl.utils import get_column_letter

_repo_x = _find_repo_root_for_extraction()
EXCEL_DIR = _resolve_table_extraction_dir(_repo_x)
export_file = EXCEL_DIR / _combinations_excel_name(MODE)

with pd.ExcelWriter(export_file, engine="openpyxl") as writer:
    combinations_df.to_excel(writer, index=False, sheet_name="Combinations")
    table_meta_df.to_excel(writer, index=False, sheet_name="Table Metadata")
    tables_with_metrics_df.to_excel(writer, index=False, sheet_name="Tables With Datasets")
    tables_with_values_df.to_excel(writer, index=False, sheet_name="Tables With Values")

    wb = writer.book
    header_fill = PatternFill(start_color="D9E1F2", end_color="D9E1F2", fill_type="solid")
    header_font = Font(bold=True)

    for ws in wb.worksheets:
        for cell in ws[1]:
            cell.font = header_font
            cell.fill = header_fill
        for col_idx, col_cells in enumerate(
            ws.iter_cols(min_row=1, max_row=ws.max_row, min_col=1, max_col=ws.max_column), start=1
        ):
            max_len = max(len(str(c.value)) if c.value is not None else 0 for c in col_cells)
            ws.column_dimensions[get_column_letter(col_idx)].width = min(max(10, max_len + 2), 60)

print(f"Excel: {export_file}")
print(f"  Combinations sheet         : {len(combinations_df)} rows")
print(f"  Metadata sheet             : {len(table_meta_df)} rows")
print(f"  Tables With Datasets sheet  : {len(tables_with_metrics_df)} rows")
print(f"  Tables With Values sheet   : {len(tables_with_values_df)} rows")
INPUT_EXCEL = export_file.resolve()
_PIPELINE_TIMING = {
    "ocr_seconds": _ocr_seconds,
    "gliner_seconds": gliner_seconds,
    "phase": "extraction_pipeline",
}
PIPELINE_TIMING_ROWS.append(_PIPELINE_TIMING)
print(f"INPUT_EXCEL = {INPUT_EXCEL}")





In [ ]:
@dataclass
class PaperMatch:
    ours_paper_norm: str
    pwc_paper_norm: str
    score: float
    method: str


def match_papers(ours_papers: list[str], pwc_papers: list[str], threshold: float) -> list[PaperMatch]:
    pwc_set = set(pwc_papers)
    matches: list[PaperMatch] = []
    for p in sorted(set(ours_papers)):
        if p in pwc_set:
            matches.append(PaperMatch(p, p, 1.0, "exact"))
            continue
        best_name, best_score = "", 0.0
        for q in pwc_set:
            score = difflib.SequenceMatcher(None, p, q).ratio()
            if score > best_score:
                best_score, best_name = score, q
        if best_name and best_score >= threshold:
            matches.append(PaperMatch(p, best_name, best_score, "fuzzy"))
        else:
            matches.append(PaperMatch(p, "", best_score, "unmatched"))
    return matches

In [ ]:
def evaluate(ours_df: pd.DataFrame, pwc_df: pd.DataFrame, threshold: float) -> dict[str, pd.DataFrame]:
    ours_papers = sorted(ours_df["paper_norm"].unique().tolist())
    pwc_papers = sorted(pwc_df["paper_norm"].unique().tolist())
    paper_matches = match_papers(ours_papers, pwc_papers, threshold=threshold)
    paper_map = {m.ours_paper_norm: m.pwc_paper_norm for m in paper_matches if m.pwc_paper_norm}

    paper_matches_df = pd.DataFrame({
        "ours_paper_norm": [m.ours_paper_norm for m in paper_matches],
        "pwc_paper_norm": [m.pwc_paper_norm for m in paper_matches],
        "match_method": [m.method for m in paper_matches],
        "match_score": [round(float(m.score), 4) for m in paper_matches],
    }).sort_values(["match_method", "match_score", "ours_paper_norm"], ascending=[True, False, True])

    per_paper_rows = []
    tp_total = fp_total = fn_total = 0
    for ours_p in ours_papers:
        ours_metrics = set(ours_df.loc[ours_df["paper_norm"] == ours_p, "metric_norm"].tolist())
        pwc_p = paper_map.get(ours_p, "")
        pwc_metrics = set(pwc_df.loc[pwc_df["paper_norm"] == pwc_p, "metric_norm"].tolist()) if pwc_p else set()
        tp = len(ours_metrics & pwc_metrics)
        fp = len(ours_metrics - pwc_metrics)
        fn = len(pwc_metrics - ours_metrics)
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = safe_f1(precision, recall)
        tp_total += tp
        fp_total += fp
        fn_total += fn
        per_paper_rows.append({
            "paper_norm": ours_p,
            "matched_pwc_paper_norm": pwc_p,
            "num_pred_metrics": len(ours_metrics),
            "num_gt_metrics": len(pwc_metrics),
            "tp": tp, "fp": fp, "fn": fn,
            "precision": round(precision, 4), "recall": round(recall, 4), "f1": round(f1, 4),
            "pred_only_metrics": ", ".join(sorted(ours_metrics - pwc_metrics)),
            "gt_only_metrics": ", ".join(sorted(pwc_metrics - ours_metrics)),
        })

    per_paper_df = pd.DataFrame(per_paper_rows).sort_values(
        ["f1", "recall", "precision"], ascending=[True, True, True]
    )

    micro_p = tp_total / (tp_total + fp_total) if (tp_total + fp_total) else 0.0
    micro_r = tp_total / (tp_total + fn_total) if (tp_total + fn_total) else 0.0
    micro_f1 = safe_f1(micro_p, micro_r)
    macro_p = float(per_paper_df["precision"].mean()) if len(per_paper_df) else 0.0
    macro_r = float(per_paper_df["recall"].mean()) if len(per_paper_df) else 0.0
    macro_f1 = float(per_paper_df["f1"].mean()) if len(per_paper_df) else 0.0

    matched_count = int((paper_matches_df["match_method"] != "unmatched").sum())
    unmatched_ours = sorted(set(ours_papers) - set(paper_map.keys()))
    unmatched_pwc = sorted(set(pwc_papers) - set(paper_map.values()))

    global_df = pd.DataFrame([
        {"metric": "micro_precision", "value": round(micro_p, 4)},
        {"metric": "micro_recall", "value": round(micro_r, 4)},
        {"metric": "micro_f1", "value": round(micro_f1, 4)},
        {"metric": "macro_precision", "value": round(macro_p, 4)},
        {"metric": "macro_recall", "value": round(macro_r, 4)},
        {"metric": "macro_f1", "value": round(macro_f1, 4)},
        {"metric": "papers_ours_total", "value": len(ours_papers)},
        {"metric": "papers_pwc_total", "value": len(pwc_papers)},
        {"metric": "papers_matched", "value": matched_count},
        {"metric": "papers_unmatched_ours", "value": len(unmatched_ours)},
        {"metric": "papers_unmatched_pwc", "value": len(unmatched_pwc)},
        {"metric": "tp_total", "value": tp_total},
        {"metric": "fp_total", "value": fp_total},
        {"metric": "fn_total", "value": fn_total},
    ])

    err_rows = []
    for _, row in per_paper_df.iterrows():
        for m in filter(None, [x.strip() for x in str(row["pred_only_metrics"]).split(",")]):
            err_rows.append({"paper_norm": row["paper_norm"], "error_type": "FP_metric", "metric": m})
        for m in filter(None, [x.strip() for x in str(row["gt_only_metrics"]).split(",")]):
            err_rows.append({"paper_norm": row["paper_norm"], "error_type": "FN_metric", "metric": m})

    return {
        "paper_matches": paper_matches_df,
        "per_paper_scores": per_paper_df,
        "global_scores": global_df,
        "errors": pd.DataFrame(err_rows),
        "unmatched_ours": pd.DataFrame({"paper_norm": unmatched_ours}),
        "unmatched_pwc": pd.DataFrame({"paper_norm": unmatched_pwc}),
    }


def enrich_ours_combinations_with_dataset_json(
    ours_df: pd.DataFrame,
    json_path: Path,
    *,
    match_threshold: float | None = None,
) -> pd.DataFrame:
    """Add `paper_match_key` aligned with GT: PDF stem / arxiv id → `_dataset_json_identity`."""
    if not json_path.is_file():
        raise FileNotFoundError(json_path)
    thr = MATCH_THRESHOLD if match_threshold is None else match_threshold
    with open(json_path, encoding="utf-8") as f:
        papers = json.load(f)
    if not isinstance(papers, list):
        raise ValueError(f"Expected a JSON list in {json_path}")

    by_arxiv, _by_slug, by_title, by_pdf_stem, by_title_stem = _pwc_final_lookup_maps(papers)
    out = ours_df.copy()
    keys: list[str] = []
    kinds: list[str] = []
    for _, row in out.iterrows():
        raw = str(row.get("paper_raw", "") or "").strip()
        obj, _how = _match_pwc_entry_for_pdf_stem(
            raw, by_arxiv, by_title, by_pdf_stem, thr, by_title_stem=by_title_stem
        )
        if obj is not None:
            mk, kind = _dataset_json_identity(obj)
        else:
            mk, kind = _pwc_identity_from_row(None, raw)
        keys.append(mk)
        kinds.append(kind)
    out["paper_match_key"] = keys
    out["paper_match_kind"] = kinds
    return out


def evaluate_combinations_id_aware(
    ours_df: pd.DataFrame,
    pwc_df: pd.DataFrame,
    threshold_title_fallback: float,
) -> dict[str, pd.DataFrame]:
    """Compare metric names grouped by `paper_match_key` (arxiv:/slug:/title:) instead of title-only fuzzy match."""
    if "paper_match_key" not in ours_df.columns or "paper_match_key" not in pwc_df.columns:
        raise ValueError("Expected column paper_match_key in both DataFrames.")

    def _agg_dataset_set(s: pd.Series) -> set[str]:
        out: set[str] = set()
        for x in s.dropna().tolist():
            if str(x).strip():
                out.add(str(x))
        return out

    ours_g = ours_df.groupby("paper_match_key", sort=False)["dataset_norm"].apply(_agg_dataset_set)
    pwc_g = pwc_df.groupby("paper_match_key", sort=False)["dataset_norm"].apply(_agg_dataset_set)

    pwc_key_list = list(pwc_g.index)
    used_pwc: set[str] = set()
    key_map: dict[str, str] = {}
    for ok in ours_g.index:
        if ok in pwc_g.index:
            key_map[ok] = ok
            used_pwc.add(ok)
            continue
        if not str(ok).startswith("title:"):
            key_map[ok] = ""
            continue
        best_pk, best_sc = "", 0.0
        for pk in pwc_key_list:
            if pk in used_pwc:
                continue
            if not str(pk).startswith("title:"):
                continue
            sc = difflib.SequenceMatcher(None, ok, pk).ratio()
            if sc > best_sc:
                best_sc, best_pk = sc, pk
        if best_pk and best_sc >= threshold_title_fallback:
            key_map[ok] = best_pk
            used_pwc.add(best_pk)
        else:
            key_map[ok] = ""

    rep_norm = ours_df.groupby("paper_match_key", sort=False)["paper_norm"].first()

    per_paper_rows = []
    tp_total = fp_total = fn_total = 0
    for ok in ours_g.index:
        ours_m = set(ours_g[ok])
        pk = key_map.get(ok, "")
        pwc_m = set(pwc_g[pk]) if pk and pk in pwc_g.index else set()
        tp = len(ours_m & pwc_m)
        fp = len(ours_m - pwc_m)
        fn = len(pwc_m - ours_m)
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = safe_f1(precision, recall)
        tp_total += tp
        fp_total += fp
        fn_total += fn
        if pk and pk in pwc_g.index:
            how = "exact_key" if ok == pk else "fuzzy_title_key"
        else:
            how = "unmatched"
        per_paper_rows.append(
            {
                "paper_norm": rep_norm.get(ok, ""),
                "ours_match_key": ok,
                "matched_pwc_match_key": pk,
                "match_kind": how,
                "num_pred_metrics": len(ours_m),
                "num_gt_metrics": len(pwc_m),
                "tp": tp,
                "fp": fp,
                "fn": fn,
                "precision": round(precision, 4),
                "recall": round(recall, 4),
                "f1": round(f1, 4),
                "pred_only_metrics": ", ".join(sorted(ours_m - pwc_m)),
                "gt_only_metrics": ", ".join(sorted(pwc_m - ours_m)),
            }
        )

    per_paper_df = pd.DataFrame(per_paper_rows).sort_values(
        ["f1", "recall", "precision"], ascending=[True, True, True]
    )

    micro_p = tp_total / (tp_total + fp_total) if (tp_total + fp_total) else 0.0
    micro_r = tp_total / (tp_total + fn_total) if (tp_total + fn_total) else 0.0
    micro_f1 = safe_f1(micro_p, micro_r)
    macro_p = float(per_paper_df["precision"].mean()) if len(per_paper_df) else 0.0
    macro_r = float(per_paper_df["recall"].mean()) if len(per_paper_df) else 0.0
    macro_f1 = float(per_paper_df["f1"].mean()) if len(per_paper_df) else 0.0

    matched_count = int((per_paper_df["match_kind"] != "unmatched").sum())
    ours_keys = set(ours_g.index)
    pwc_keys = set(pwc_g.index)
    mapped_pwc = {key_map[k] for k in ours_keys if key_map.get(k)}
    unmatched_ours_keys = sorted(k for k in ours_keys if not key_map.get(k))

    paper_matches_df = pd.DataFrame(
        {
            "ours_paper_norm": per_paper_df["paper_norm"],
            "ours_match_key": per_paper_df["ours_match_key"],
            "pwc_match_key": per_paper_df["matched_pwc_match_key"],
            "match_method": per_paper_df["match_kind"],
            "match_score": per_paper_df["match_kind"].map(lambda x: 1.0 if x == "exact_key" else (0.9 if x == "fuzzy_title_key" else 0.0)),
        }
    )

    global_df = pd.DataFrame(
        [
            {"metric": "eval_mode", "value": "arxiv_slug_title_keys"},
            {"metric": "micro_precision", "value": round(micro_p, 4)},
            {"metric": "micro_recall", "value": round(micro_r, 4)},
            {"metric": "micro_f1", "value": round(micro_f1, 4)},
            {"metric": "macro_precision", "value": round(macro_p, 4)},
            {"metric": "macro_recall", "value": round(macro_r, 4)},
            {"metric": "macro_f1", "value": round(macro_f1, 4)},
            {"metric": "papers_ours_total", "value": len(ours_keys)},
            {"metric": "papers_pwc_total", "value": len(pwc_keys)},
            {"metric": "papers_matched", "value": matched_count},
            {"metric": "papers_unmatched_ours", "value": int(per_paper_df["match_kind"].eq("unmatched").sum())},
            {"metric": "papers_unmatched_pwc", "value": len(pwc_keys - mapped_pwc)},
            {"metric": "tp_total", "value": tp_total},
            {"metric": "fp_total", "value": fp_total},
            {"metric": "fn_total", "value": fn_total},
        ]
    )

    err_rows = []
    for _, row in per_paper_df.iterrows():
        for m in filter(None, [x.strip() for x in str(row["pred_only_metrics"]).split(",")]):
            err_rows.append({"paper_norm": row["paper_norm"], "error_type": "FP_metric", "metric": m})
        for m in filter(None, [x.strip() for x in str(row["gt_only_metrics"]).split(",")]):
            err_rows.append({"paper_norm": row["paper_norm"], "error_type": "FN_metric", "metric": m})

    return {
        "paper_matches": paper_matches_df,
        "per_paper_scores": per_paper_df,
        "global_scores": global_df,
        "errors": pd.DataFrame(err_rows),
        "unmatched_ours": pd.DataFrame({"paper_match_key": unmatched_ours_keys}),
        "unmatched_pwc": pd.DataFrame({"paper_match_key": sorted(pwc_keys - mapped_pwc)}),
    }


## Text-only predictions (TEI + GLiNER)

`text_only` runs GLiNER on GROBID TEI narrative text (`data/xml_files_3/`). Evaluation ground truth is **`data/pwc_final.json`** only (Datasets field).


In [ ]:

_KGE_MODEL_DATASET_BLOCKLIST = frozenset({
    "transe", "distmult", "complex", "rotate", "conve", "tucker", "rescal",
    "analogy", "simple", "hole", "transh", "transr", "transd", "mtransh",
    "quate", "pairre", "crosses", "tntcomplex", "convtranse", "rotate3d",
})


def _is_plausible_dataset_norm(nm: str) -> bool:
    s = str(nm or "").strip().lower()
    if not s or s in _KGE_MODEL_DATASET_BLOCKLIST:
        return False
    return True


def _tei_narrative_chunks_from_path(
    xml_path: Path,
    section_keywords: list[str],
    scan_full_body: bool,
    max_chars: int | None = None,
) -> list[str]:
    """Narrative TEI chunks (no <table> cells) for GLiNER text_only."""
    from bs4 import BeautifulSoup

    limit = max_chars if max_chars is not None else GLINER2_MAX_CHARS
    with open(xml_path, encoding="utf-8", errors="replace") as f:
        soup = BeautifulSoup(f, "lxml-xml")
    body = soup.find("body")
    if body is None:
        return []
    keywords = [k.lower() for k in section_keywords if k]
    if not keywords:
        body_full = BeautifulSoup(str(body), "lxml-xml").find("body") or body
        for table in body_full.find_all("table"):
            table.decompose()
        txt = body_full.get_text(separator=" ", strip=True)
        if not txt:
            return []
        return [txt[i : i + limit] for i in range(0, len(txt), limit)]
    chunks: list[str] = []
    for div in body.find_all("div"):
        head = div.find("head")
        if not head:
            continue
        title = head.get_text(strip=True).lower()
        if keywords and not any(kw in title for kw in keywords):
            continue
        txt = div.get_text(separator=" ", strip=True)
        if txt:
            chunks.append(txt[:limit])
    if scan_full_body and not chunks:
        txt = body.get_text(separator=" ", strip=True)
        if txt:
            chunks.append(txt[:limit])
    return chunks


def _resolve_xml_for_pdf_stem(
    repo_root: Path,
    pdf_stem: str,
    *,
    xml_rel_dir: Path | None = None,
) -> Path | None:
    """Resolve GROBID XML for a PDF stem (xml_files_3: <stem>.tei.xml)."""
    stem = str(pdf_stem).strip()
    if not stem:
        return None
    xml_dir = (repo_root / (xml_rel_dir or Path("data/xml_files_3"))).resolve()
    if not xml_dir.is_dir():
        return None
    direct = xml_dir / f"{stem}.tei.xml"
    if direct.is_file():
        return direct
    a = _arxiv_norm_from_text(stem)
    if a:
        hits = sorted(xml_dir.glob(f"*{a}*.xml"))
        if hits:
            return hits[0].resolve()
    return None


def _gliner_datasets_for_paper(chunks: list[str]) -> list[str]:
    if "gliner2_model" not in globals():
        raise RuntimeError("gliner2_model not loaded")
    paper_sets: dict[str, set[str]] = {k: set() for k in GLINER2_LABELS}
    for chunk in chunks:
        if not chunk:
            continue
        ents = _extract_entities(str(chunk))
        for label in GLINER2_LABELS:
            for v in ents.get(label, []) or []:
                if v:
                    paper_sets[label].add(str(v).strip())
    kept_norm: set[str] = set()
    kept_raw: list[str] = []
    for d in paper_sets.get("dataset", set()):
        nm = normalize_dataset(d)
        if not nm or not _is_plausible_dataset_norm(nm):
            continue
        if nm not in kept_norm:
            kept_norm.add(nm)
            kept_raw.append(d)
    return sorted(kept_raw, key=lambda x: normalize_dataset(x))


def build_text_only_pred_df_gliner_tei(
    repo_root: Path,
    *,
    json_basename: str,
    xml_rel_dir: Path,
    section_keywords: list[str],
    scan_full_body: bool,
    match_threshold: float,
    pdf_paths: list | None = None,
    pdf_rel_dir: Path | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """text_only: GLiNER on GROBID TEI narrative (no GT vocabulary filtering)."""
    if "gliner2_model" not in globals():
        raise RuntimeError("gliner2_model not loaded — run GLiNER extraction cells first.")
    jp = (repo_root / "data" / json_basename).resolve()
    with open(jp, encoding="utf-8") as f:
        papers = json.load(f)
    by_arxiv, _by_slug, by_title, by_pdf_stem, by_title_stem = _pwc_final_lookup_maps(papers)
    if pdf_paths is None:
        pdf_paths = sorted((repo_root / (pdf_rel_dir or PDF_FILES_DIR)).glob("*.pdf"))
    pred_rows: list[dict] = []
    audit_rows: list[dict] = []
    for pdf_path in pdf_paths:
        stem = Path(pdf_path).stem
        obj, match_how = _match_pwc_entry_for_pdf_stem(
            stem, by_arxiv, by_title, by_pdf_stem, match_threshold,
            by_title_stem=by_title_stem,
        )
        if obj is None or not _pwc_datasets_list(obj):
            continue
        xml_path = _resolve_xml_for_pdf_stem(repo_root, stem, xml_rel_dir=xml_rel_dir)
        pn = normalize_paper(obj.get("title") or stem.replace("_", " "))
        chunks: list[str] = []
        if xml_path:
            chunks.extend(
                _tei_narrative_chunks_from_path(xml_path, section_keywords, scan_full_body)
            )
        dsets = _gliner_datasets_for_paper(chunks)
        audit_rows.append({
            "paper_raw": stem,
            "paper_norm": pn,
            "pwc_match": match_how,
            "has_xml": xml_path is not None,
            "n_text_datasets": len(dsets),
            "text_sources": "gliner_tei_narrative" if chunks else "",
            "text_datasets_flat": ", ".join(dsets),
        })
        for d in dsets:
            pred_rows.append({
                "paper_raw": stem,
                "dataset_raw": d,
                "metric_raw": "",
                "paper_norm": pn,
                "dataset_norm": normalize_dataset(d),
                "metric_norm": "",
            })
    pred_df = pd.DataFrame(pred_rows).drop_duplicates() if pred_rows else pd.DataFrame(columns=_PRED_DF_COLS)
    print(
        f"[text_only GLiNER] pred rows={len(pred_df)} | papers with >=1 dataset="
        f"{int(sum(1 for r in audit_rows if r.get('n_text_datasets', 0) > 0))} / {len(audit_rows)}"
    )
    return pred_df, pd.DataFrame(audit_rows)


print("Text-only helpers ready (TEI + GLiNER; GT = pwc_final.json only).")


## Evaluation vs `pwc_final.json` (`pdf_files_3`)

Compares extraction to **Datasets** in `data/pwc_final.json` for PDFs in the selected corpus (papers with non-empty dataset lists only).

Writes `gliner_dataset_eval_allpapers.xlsx` and `gliner_dataset_audit_allpapers.xlsx` (three modes). See `README_pwc_benchmarks.md` for execution order.


In [ ]:
if EVALUATE_COMBINATIONS_VS_PWC and results is not None:
    display(results["global_scores"])
    display(results["per_paper_scores"].head(25))
    display(results["paper_matches"].head(25))
else:
    print("PwC per-paper tables skipped (EVALUATE_COMBINATIONS_VS_PWC=False).")

In [ ]:
# Diagnostics: corpus PDFs vs Combinations papers
from pathlib import Path
import pandas as pd
import re
import unicodedata


def _strip_accents_local(text: str) -> str:
    return "".join(ch for ch in unicodedata.normalize("NFKD", text) if not unicodedata.combining(ch))


def _normalize_paper_local(text: object) -> str:
    if text is None:
        return ""
    s = str(text).strip().lower()
    s = _strip_accents_local(s)
    if s.startswith("http"):
        s = s.rstrip("/").split("/")[-1]
    s = re.sub(r"[^a-z0-9]+", " ", s).strip()
    return s


repo_root = _find_repo_root_for_extraction() if "_find_repo_root_for_extraction" in globals() else Path.cwd().resolve()

# 1) Resolve Combinations Excel (table_extraction/) + PDF folder (data/pdf_files)
if "INPUT_EXCEL" in globals() and Path(INPUT_EXCEL).exists():
    comb_xlsx = Path(INPUT_EXCEL)
else:
    mode = MODE if "MODE" in globals() else "filtered"
    comb_xlsx = repo_root / "table_extraction" / f"gliner2_lightonocr_dataset_combinations_{mode}.xlsx"
    if not comb_xlsx.is_file():
        comb_xlsx = None

if comb_xlsx is None:
    raise FileNotFoundError("Combinations Excel not found in table_extraction/.")

pdf_dir = _resolve_pdf_files_dir(repo_root) if "_resolve_pdf_files_dir" in globals() else (repo_root / PDF_FILES_DIR)
if not pdf_dir.is_dir():
    raise FileNotFoundError(f"PDF folder not found: {pdf_dir}")

# 3) Load Combinations sheet
comb = pd.read_excel(comb_xlsx, sheet_name="Combinations")
if "paper" not in comb.columns:
    raise ValueError(f"Combinations sheet has no 'paper' column. Columns: {list(comb.columns)}")

pdf_stems_raw = sorted({p.stem for p in pdf_dir.glob("*.pdf")})
pdf_norm = {_normalize_paper_local(s): s for s in pdf_stems_raw if _normalize_paper_local(s)}

papers_raw = comb["paper"].dropna().astype(str).tolist()
papers_norm_set = {_normalize_paper_local(x) for x in papers_raw if _normalize_paper_local(x)}

missing_norm = sorted(set(pdf_norm.keys()) - papers_norm_set)
present_norm = sorted(set(pdf_norm.keys()) & papers_norm_set)

missing_df = pd.DataFrame({
    "missing_pdf_stem": [pdf_norm[n] for n in missing_norm],
    "missing_pdf_norm": missing_norm,
})

present_df = pd.DataFrame({
    "present_pdf_stem": [pdf_norm[n] for n in present_norm],
    "present_pdf_norm": present_norm,
})

print("Excel Combinations:", comb_xlsx)
print("PDF folder used:", pdf_dir)
print("PDFs totales:", len(pdf_stems_raw))
print("Unique papers in Combinations (normalized):", len(papers_norm_set))
print("PDFs present in Combinations:", len(present_norm))
print("PDFs missing from Combinations:", len(missing_norm))

print("\n=== First missing (up to 50) ===")
display(missing_df.head(50))

print("\n=== Sample of present (up to 20) ===")
display(present_df.head(20))

if EXPORT_COVERAGE_MISSING_TO_EXCEL:
    print("\nNote: this check is also written to the 'coverage_missing' sheet in the main Excel report.")
else:
    print("\nNote: coverage gap is printed here only (EXPORT_COVERAGE_MISSING_TO_EXCEL=False).")

In [ ]:
# Full-corpus audit: GLiNER on LightOnOCR tables + three-mode evaluation (BERTScore)
from pathlib import Path
import pandas as pd
import time
import difflib
import torch
from bert_score import score as bert_score_fn

_BERTSCORE_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
_BERTSCORE_LANG = globals().get("BERTSCORE_LANG", "en")

_eval_suffix = globals().get("EVAL_OUTPUT_SUFFIX", "_minimal")
USE_BLACKLIST = False

AUDIT_ALL_OUTPUT_XLSX = _resolve_table_extraction_dir(_find_repo_root_for_extraction()) / f"gliner_dataset_audit_allpapers{_eval_suffix}.xlsx"
EVAL_ALL_OUTPUT_XLSX = _resolve_table_extraction_dir(_find_repo_root_for_extraction()) / f"gliner_dataset_eval_allpapers{_eval_suffix}.xlsx"
REEXTRACT_TABLES_WITH_GLINER = True  # False => use Tables With Datasets from INPUT_EXCEL only

_repo_all = _find_repo_root_for_extraction()
_pdf_rel_all = PDF_FILES_DIR if isinstance(PDF_FILES_DIR, Path) else Path(PDF_FILES_DIR)
if _pdf_rel_all.is_absolute():
    _pdf_rel_all = _pdf_rel_all.relative_to(_repo_all)

_gt_json_name_all = globals().get("DATASET_METRICS_GT_JSON_BASENAME") or globals().get("PWC_GT_JSON") or "pwc_final.json"
_match_thr_all = float(globals().get("MATCH_THRESHOLD", 0.5))
_gt_json_path = (_repo_all / "data" / _gt_json_name_all).resolve()

_gt_df_all, _gt_audit_all, _gt_corpus_all = build_ground_truth_for_pdf_corpus(
    _repo_all,
    _gt_json_name_all,
    _pdf_rel_all,
    match_threshold=_match_thr_all,
    skip_empty_datasets=True,
)


def _df_to_pred_dataset(df_like: pd.DataFrame) -> pd.DataFrame:
    cols = ["paper_raw", "dataset_raw", "metric_raw", "paper_norm", "dataset_norm", "metric_norm"]
    if df_like is None or df_like.empty:
        return pd.DataFrame(columns=cols)
    rows = []
    for _, r in df_like.iterrows():
        paper = str(r.get("paper", "") or r.get("paper_raw", "")).strip()
        dataset_raw = str(r.get("dataset", "") or r.get("dataset_raw", "")).strip()
        ds = normalize_dataset(dataset_raw)
        if not paper or not ds:
            continue
        rows.append({
            "paper_raw": paper,
            "dataset_raw": dataset_raw,
            "metric_raw": str(r.get("metric", "") or r.get("metric_raw", "")).strip(),
            "paper_norm": normalize_paper(paper),
            "dataset_norm": ds,
            "metric_norm": "",
        })
    out = pd.DataFrame(rows)
    if out.empty:
        return pd.DataFrame(columns=cols)
    out = out[(out["paper_norm"] != "") & (out["dataset_norm"] != "")]
    return out.drop_duplicates()


def _merge_tables_and_text_minimal(
    tables_df: pd.DataFrame,
    text_df: pd.DataFrame,
) -> pd.DataFrame:
    """Naive union of tables_only and text_only predictions (no GT oracle)."""
    cols = ["paper_raw", "dataset_raw", "metric_raw", "paper_norm", "dataset_norm", "metric_norm"]
    parts = []
    if tables_df is not None and not tables_df.empty:
        parts.append(tables_df)
    if text_df is not None and not text_df.empty:
        parts.append(text_df)
    if not parts:
        return pd.DataFrame(columns=cols)
    return pd.concat(parts, ignore_index=True).drop_duplicates()


def _agg_dataset_list(s: pd.Series) -> list[str]:
    seen: set[str] = set()
    out: list[str] = []
    for x in s.dropna().tolist():
        sx = str(x).strip()
        if sx and sx not in seen:
            seen.add(sx)
            out.append(sx)
    return out


def _bertscore_for_dataset_lists(
    pred_list: list[str],
    ref_list: list[str],
) -> tuple[float, float, float]:
    """P_bert, R_bert, F1_bert for one paper (same protocol as experimentation_utils.calcular_bertscore_listas)."""
    preds = [str(x).strip() for x in pred_list if str(x).strip()]
    refs = [str(x).strip() for x in ref_list if str(x).strip()]
    if not preds and not refs:
        return 1.0, 1.0, 1.0
    if not preds or not refs:
        return 0.0, 0.0, 0.0
    cands = [" ".join(preds)]
    references = [" ".join(refs)]
    P, R, F1 = bert_score_fn(
        cands, references, lang=_BERTSCORE_LANG, device=_BERTSCORE_DEVICE, verbose=False
    )
    return float(P[0]), float(R[0]), float(F1[0])


def _id_aware_bertscore_breakdown(
    pred_df: pd.DataFrame, gt_df: pd.DataFrame, gt_json_path: Path, thr: float,
    *, value_col: str = "metric_norm", agg_fn=None,
):
    """Id-aware matching: macro BERTScore + exact-match counts (n_intersection)."""
    if agg_fn is None:
        agg_fn = _agg_metric_list if value_col == "metric_norm" else _agg_dataset_list
    cols = ["paper_raw", "dataset_raw", "metric_raw", "paper_norm", "dataset_norm", "metric_norm"]
    pred_df = pred_df if pred_df is not None else pd.DataFrame(columns=cols)
    pred_e = enrich_ours_combinations_with_dataset_json(pred_df, gt_json_path, match_threshold=thr)

    ours_g = (
        pred_e.groupby("paper_match_key", sort=False)[value_col].apply(agg_fn)
        if len(pred_e) else pd.Series(dtype=object)
    )
    gt_g = (
        gt_df.groupby("paper_match_key", sort=False)[value_col].apply(agg_fn)
        if len(gt_df) else pd.Series(dtype=object)
    )

    gt_key_to_title = gt_df.groupby("paper_match_key", sort=False)["paper_raw"].first().to_dict() if len(gt_df) else {}
    pwc_key_list = list(gt_g.index)
    used_pwc: set[str] = set()
    key_map: dict[str, str] = {}

    for ok in ours_g.index:
        if ok in gt_g.index:
            key_map[ok] = ok
            used_pwc.add(ok)
            continue
        if not str(ok).startswith("title:"):
            key_map[ok] = ""
            continue
        best_pk, best_sc = "", 0.0
        for pk in pwc_key_list:
            if pk in used_pwc or not str(pk).startswith("title:"):
                continue
            sc = difflib.SequenceMatcher(None, str(ok), str(pk)).ratio()
            if sc > best_sc:
                best_sc, best_pk = sc, pk
        if best_pk and best_sc >= thr:
            key_map[ok] = best_pk
            used_pwc.add(best_pk)
        else:
            key_map[ok] = ""

    rep_norm = pred_e.groupby("paper_match_key", sort=False)["paper_norm"].first().to_dict() if len(pred_e) else {}
    all_keys = set(gt_g.index) | set(ours_g.index)
    detail_rows = []
    p_scores: list[float] = []
    r_scores: list[float] = []
    f1_scores: list[float] = []
    tp_total = 0

    pred_label = "pipeline_metrics" if value_col == "metric_norm" else "pipeline_datasets"
    gt_label = "ground_truth_metrics" if value_col == "metric_norm" else "ground_truth_datasets"

    for k in sorted(all_keys):
        ours_m = set(ours_g.get(k, [])) if k in ours_g.index else set()
        if k in ours_g.index:
            mk = key_map.get(k, "")
            gt_m = set(gt_g[mk]) if mk and mk in gt_g.index else set()
            paper_norm = rep_norm.get(k, str(k).replace("title:", "").strip())
            paper_title = gt_key_to_title.get(mk, "")
        else:
            gt_m = set(gt_g[k]) if k in gt_g.index else set()
            paper_norm = str(k).replace("title:", "").strip()
            paper_title = gt_key_to_title.get(k, "")

        if globals().get("EVAL_ONLY_PWC_MATCHED_PAPERS", True) and not gt_m:
            continue

        ours_list = sorted(ours_m)
        gt_list = sorted(gt_m)
        inter = ours_m & gt_m
        only_p = ours_m - gt_m
        only_g = gt_m - ours_m
        tp_total += len(inter)

        p_b, r_b, f1_b = (
            globals().get("_bertscore_for_item_lists")
            or globals().get("_bertscore_for_dataset_lists")
        )(ours_list, gt_list)
        p_scores.append(p_b)
        r_scores.append(r_b)
        f1_scores.append(f1_b)

        detail_rows.append({
            "paper_norm": paper_norm,
            "paper_title": paper_title,
            pred_label: ", ".join(sorted(ours_m)),
            gt_label: ", ".join(sorted(gt_m)),
            "intersection": ", ".join(sorted(inter)),
            "only_pipeline": ", ".join(sorted(only_p)),
            "only_ground_truth": ", ".join(sorted(only_g)),
        })

    n_papers = len(f1_scores)
    summary = {
        "P_bert": round(sum(p_scores) / n_papers, 4) if n_papers else 0.0,
        "R_bert": round(sum(r_scores) / n_papers, 4) if n_papers else 0.0,
        "F1_bert": round(sum(f1_scores) / n_papers, 4) if n_papers else 0.0,
        "n_pipeline": int(sum(len(set(ours_g[k])) for k in ours_g.index)) if len(ours_g) else 0,
        "n_ground_truth": int(sum(len(set(gt_g[k])) for k in gt_g.index)) if len(gt_g) else 0,
        "n_intersection": int(tp_total),
    }
    return summary, pd.DataFrame(detail_rows)



# --- 1) tables_only (re-extraccion GLiNER) o baseline desde Excel ---
_t0 = time.perf_counter()
if REEXTRACT_TABLES_WITH_GLINER:
    if "ocr_outputs" not in globals():
        raise RuntimeError("No existe ocr_outputs. Ejecuta la seccion de extraccion GLiNER primero.")
    if "_extract_entities_raw" not in globals() or "_header_and_rows_from_html" not in globals():
        raise RuntimeError("No estan cargados _extract_entities_raw/_header_and_rows_from_html. Ejecuta la celda GLiNER de extraccion.")

    rows_tables = []
    for pdf_stem in list(ocr_outputs.keys()):
        ocr_result = ocr_outputs[pdf_stem]
        paper_datasets = set()
        for page_data in ocr_result.get("results", []):
            for table in page_data.get("tables", []):
                html = table.get("html", "")
                if not html:
                    continue
                header_lines, body_lines = _header_and_rows_from_html(html)
                if not header_lines and not body_lines:
                    continue
                header_context = "\n".join(header_lines)
                table_best = {}
                if header_context:
                    _merge_best(table_best, _extract_entities_raw(header_context))
                for row_text in body_lines:
                    prompt = f"Table column headers: {header_context}\nRow: {row_text}" if header_context else row_text
                    _merge_best(table_best, _extract_entities_raw(prompt))
                if not table_best:
                    _merge_best(table_best, _extract_entities_raw("\n".join(header_lines + body_lines)))
                for value, (label, _score) in table_best.items():
                    if label != "dataset":
                        continue
                    nm = normalize_dataset(value)
                    if nm:
                        paper_datasets.add(nm)
        for ds in sorted(paper_datasets):
            rows_tables.append({
                "paper_raw": pdf_stem,
                "dataset_raw": ds,
                "metric_raw": "",
                "paper_norm": normalize_paper(pdf_stem),
                "dataset_norm": ds,
                "metric_norm": "",
            })
    _pred_tables_all = pd.DataFrame(rows_tables).drop_duplicates() if rows_tables else pd.DataFrame(
        columns=["paper_raw", "dataset_raw", "metric_raw", "paper_norm", "dataset_norm", "metric_norm"]
    )
else:
    _pred_tables_all = load_pred_tables_only_sheet(INPUT_EXCEL)

t_tables = time.perf_counter() - _t0


# --- 2) text_only (usar cache si ya existe; si no, construir) ---
_t1 = time.perf_counter()
if "_text_only_dm" in globals() and isinstance(_text_only_dm, pd.DataFrame) and not _text_only_dm.empty:
    _pred_text_all = _text_only_dm.copy()
elif "build_text_only_pred_df_gliner_tei" in globals():
    _corpus_pdfs_all = sorted((_repo_all / _pdf_rel_all).glob("*.pdf"))
    _pred_text_all, _audit_text_all = build_text_only_pred_df_gliner_tei(
        _repo_all,
        json_basename=_gt_json_name_all,
        xml_rel_dir=XML_FILES_DIR,
        section_keywords=globals().get("GT_XML_SECTION_KEYWORDS", []),
        scan_full_body=bool(globals().get("GT_SCAN_FULL_BODY_IF_EMPTY", True)),
        match_threshold=_match_thr_all,
        pdf_paths=_corpus_pdfs_all,
        pdf_rel_dir=_pdf_rel_all,
    )
else:
    _pred_text_all = pd.DataFrame(columns=["paper_raw", "dataset_raw", "metric_raw", "paper_norm", "dataset_norm", "metric_norm"])
    print("WARN: no se pudo construir text_only; faltan funciones/celdas previas.")

t_text = time.perf_counter() - _t1

_pred_tables_all = _df_to_pred_dataset(_pred_tables_all)
_pred_text_all = _df_to_pred_dataset(_pred_text_all)
_pred_union_all = _merge_tables_and_text_minimal(_pred_tables_all, _pred_text_all)


# --- 3) evaluacion BERTScore (macro avg por paper) ---
mode_inputs = [
    ("tables_only", _pred_tables_all, t_tables),
    ("text_only", _pred_text_all, t_text),
    ("tables_plus_text", _pred_union_all, t_tables + t_text),
]

summary_rows = []
audit_parts = []

for mode_name, pred_df, sec in mode_inputs:
    summ, det = _id_aware_bertscore_breakdown(pred_df, _gt_df_all, _gt_json_path, _match_thr_all, value_col="dataset_norm")
    summary_rows.append({
        "mode": mode_name,
        "P_bert": summ["P_bert"],
        "R_bert": summ["R_bert"],
        "F1_bert": summ["F1_bert"],
        "time_seconds": round(sec, 2),
        "n_pipeline": summ["n_pipeline"],
        "n_ground_truth": summ["n_ground_truth"],
        "n_intersection": summ["n_intersection"],
    })
    det = det.copy()
    det.insert(1, "mode", mode_name)
    audit_parts.append(det)

eval_df = pd.DataFrame(summary_rows)
audit_df = pd.concat(audit_parts, ignore_index=True) if audit_parts else pd.DataFrame(
    columns=["paper_norm", "paper_title", "mode", "pipeline_datasets", "ground_truth_datasets", "intersection", "only_pipeline", "only_ground_truth"]
)

audit_df = audit_df[[
    "paper_norm",
    "paper_title",
    "mode",
    "pipeline_datasets",
    "ground_truth_datasets",
    "intersection",
    "only_pipeline",
    "only_ground_truth",
]].copy()

audit_df = audit_df.rename(columns={"paper_norm": "paper", "paper_title": "title"})
_mode_order = {"tables_only": 0, "text_only": 1, "tables_plus_text": 2}
audit_df["_paper_k"] = audit_df["paper"].astype(str)
audit_df["_mode_k"] = audit_df["mode"].map(_mode_order).fillna(9)
audit_df = audit_df.sort_values(["_paper_k", "_mode_k"], kind="stable").drop(columns=["_paper_k", "_mode_k"]).reset_index(drop=True)

with pd.ExcelWriter(EVAL_ALL_OUTPUT_XLSX, engine="openpyxl") as _w:
    eval_df.to_excel(_w, sheet_name="evaluation", index=False)

with pd.ExcelWriter(AUDIT_ALL_OUTPUT_XLSX, engine="openpyxl") as _w:
    audit_df.to_excel(_w, sheet_name="audit_all_papers", index=False)

print(f"Minimal pipeline: USE_BLACKLIST={USE_BLACKLIST}, FILTER_PREDICTIONS_TO_PWC_GT_DATASETS={FILTER_PREDICTIONS_TO_PWC_GT_DATASETS}")
print(f"Eval: BERTScore macro avg per paper (lang={_BERTSCORE_LANG}, device={_BERTSCORE_DEVICE})")
print(f"  Per paper: join dataset lists with spaces → P_bert, R_bert, F1_bert; corpus = mean over papers")
print(f"Evaluation Excel written: {EVAL_ALL_OUTPUT_XLSX.resolve()}")
print(f"Audit Excel written: {AUDIT_ALL_OUTPUT_XLSX.resolve()}")
print(f"Eval rows: {len(eval_df)} | Audit rows: {len(audit_df)} | Papers: {audit_df['paper'].nunique() if len(audit_df) else 0}")
display(eval_df)
display(audit_df.head(30))
